In [ ]:
import json
from pathlib import Path

import pandas as pd

def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "processed" / "library_data" / "story_categories_v1.json").exists():
            return candidate
    raise FileNotFoundError("Could not find project root containing data/processed/library_data/story_categories_v1.json")

ROOT = find_project_root()
LIBRARY_DIR = ROOT / 'data/processed/library_data'
data_path = LIBRARY_DIR / 'story_categories_v1.json'

with data_path.open(encoding='utf-8') as f:
    stories = json.load(f)

df = pd.DataFrame(stories)

success_df = df[df['text'].fillna('').str.strip() != ''].copy()

print(f'Total successfully downloaded stories: {len(success_df)}')
print(f'Input path: {data_path.relative_to(ROOT)}')

category_level_counts = (
    success_df
    .groupby(['category_v1', 'reading_level_key'])
    .size()
    .reset_index(name='book_count')
    .sort_values(['category_v1', 'reading_level_key'])
)

display(category_level_counts)

category_level_pivot = (
    category_level_counts
    .pivot(index='category_v1', columns='reading_level_key', values='book_count')
    .fillna(0)
    .astype(int)
)

display(category_level_pivot)


Total successfully downloaded stories: 7994


,category_v1,reading_level_key,book_count
0,Animals,level_1,1191
1,Animals,level_2,1106
2,Animals,level_3,837
3,Daily Life,level_1,1345
4,Daily Life,level_2,1493
5,Daily Life,level_3,1533
6,Science & Knowledge,level_1,109
7,Science & Knowledge,level_2,136
8,Science & Knowledge,level_3,244


reading_level_key,level_1,level_2,level_3
category_v1,,,
Animals,1191,1106,837
Daily Life,1345,1493,1533
Science & Knowledge,109,136,244


In [ ]:
# Count the number of articles with 50 words or fewer and those with more than 2500 words
short_count = (success_df['word_count'] < 50).sum()
long_count = (success_df['word_count'] > 2500).sum()

summary_df = pd.DataFrame(
    {
        'range': ['< 50 words', '> 2500 words'],
        'book_count': [short_count, long_count],
    }
)

display(summary_df)

print(f'Stories with fewer than 50 words: {short_count}')
print(f'Stories with more than 2500 words: {long_count}')


,range,book_count
0,< 50 words,629
1,> 2500 words,66


Stories with fewer than 50 words: 629
Stories with more than 2500 words: 66


In [7]:
# Remove articles with fewer than 50 words and more than 2500 words, save the cleaned file, and recalculate the quantity of each level for each category.
output_path = LIBRARY_DIR / 'story_categories_v2.json'

clean_df = success_df[(success_df['word_count'] >= 50) & (success_df['word_count'] <= 2500)].copy()

print(f'Total stories after length filtering: {len(clean_df)}')

clean_records = clean_df.to_dict(orient='records')

with output_path.open('w', encoding='utf-8') as f:
    json.dump(clean_records, f, ensure_ascii=False, indent=2)

print(f'Saved cleaned file to {output_path}')

clean_category_level_counts = (
    clean_df
    .groupby(['category_v1', 'reading_level_key'])
    .size()
    .reset_index(name='book_count')
    .sort_values(['category_v1', 'reading_level_key'])
)

display(clean_category_level_counts)

clean_category_level_pivot = (
    clean_category_level_counts
    .pivot(index='category_v1', columns='reading_level_key', values='book_count')
    .fillna(0)
    .astype(int)
)

display(clean_category_level_pivot)


Total stories after length filtering: 7299
Saved cleaned file to story_categories_v2.json


,category_v1,reading_level_key,book_count
0,Animals,level_1,1010
1,Animals,level_2,1057
2,Animals,level_3,795
3,Daily Life,level_1,1154
4,Daily Life,level_2,1430
5,Daily Life,level_3,1431
6,Science & Knowledge,level_1,84
7,Science & Knowledge,level_2,117
8,Science & Knowledge,level_3,221


reading_level_key,level_1,level_2,level_3
category_v1,,,
Animals,1010,1057,795
Daily Life,1154,1430,1431
Science & Knowledge,84,117,221


In [9]:
# Check for duplicate or similar titles in V2
# Read from story_categories_v2.json instead of using clean_df directly
from difflib import SequenceMatcher

v2_path = LIBRARY_DIR / 'story_categories_v2.json'

with v2_path.open(encoding='utf-8') as f:
    v2_stories = json.load(f)

title_check_df = pd.DataFrame(v2_stories)[['unified_id', 'title', 'category_v1', 'reading_level_key']].copy()
title_check_df['normalized_title'] = (
    title_check_df['title']
    .fillna('')
    .str.lower()
    .str.replace(r'[^a-z0-9]+', ' ', regex=True)
    .str.strip()
)

# 1. Exact duplicate titles after normalization
exact_duplicate_titles = (
    title_check_df
    .groupby('normalized_title')
    .filter(lambda x: x['normalized_title'].iloc[0] != '' and len(x) > 1)
    .sort_values(['normalized_title', 'title'])
)

print(f'Exact duplicate title groups: {exact_duplicate_titles["normalized_title"].nunique()}')
display(exact_duplicate_titles[['unified_id', 'title', 'normalized_title', 'category_v1', 'reading_level_key']].head(50))

# 2. Similar titles: compare only within small buckets
unique_titles = (
    title_check_df[['title', 'normalized_title']]
    .drop_duplicates()
    .query("normalized_title != ''")
    .reset_index(drop=True)
)

def make_title_key(text):
    words = text.split()
    first_word = words[0] if words else ''
    prefix = text[:8]
    length_bucket = len(text) // 5
    return (first_word, prefix, length_bucket)

unique_titles['title_key'] = unique_titles['normalized_title'].apply(make_title_key)

similar_pairs = []
threshold = 0.9

for _, group in unique_titles.groupby('title_key'):
    group = group.reset_index(drop=True)
    if len(group) < 2:
        continue

    for i in range(len(group)):
        for j in range(i + 1, len(group)):
            t1 = group.loc[i, 'normalized_title']
            t2 = group.loc[j, 'normalized_title']

            if abs(len(t1) - len(t2)) > 8:
                continue

            score = SequenceMatcher(None, t1, t2).ratio()
            if score >= threshold and t1 != t2:
                similar_pairs.append(
                    {
                        'title_1': group.loc[i, 'title'],
                        'title_2': group.loc[j, 'title'],
                        'normalized_title_1': t1,
                        'normalized_title_2': t2,
                        'similarity': round(score, 3),
                    }
                )

similar_titles_df = (
    pd.DataFrame(similar_pairs)
    .sort_values('similarity', ascending=False)
    .drop_duplicates(subset=['title_1', 'title_2'])
    if similar_pairs else pd.DataFrame()
)

print(f'Similar title pairs (similarity >= {threshold}): {len(similar_titles_df)}')
display(similar_titles_df.head(50))


Exact duplicate title groups: 324


,unified_id,title,normalized_title,category_v1,reading_level_key
838,storyweaver_24725,A Beautiful Day,a beautiful day,Animals,level_1
279,storyweaver_6991,A beautiful day,a beautiful day,Animals,level_3
6011,storyweaver_572080,A Call,a call,Daily Life,level_2
6015,storyweaver_573048,A Call,a call,Daily Life,level_2
5342,storyweaver_483232,A Carriage Ride to the West,a carriage ride to the west,Daily Life,level_3
5301,storyweaver_480138,A Carriage Ride to the West.,a carriage ride to the west,Daily Life,level_3
5537,storyweaver_505006,A Carriage Ride to the West.,a carriage ride to the west,Daily Life,level_3
1739,storyweaver_84929,A Day at the Park,a day at the park,Daily Life,level_1
359,storyweaver_9765,A day at the park,a day at the park,Daily Life,level_2
4328,storyweaver_346015,A day at the park,a day at the park,Daily Life,level_3


Similar title pairs (similarity >= 0.9): 349


,title_1,title_2,normalized_title_1,normalized_title_2,similarity
340,There goes Poli the Pufferfish,There goes Polli the Pufferfish,there goes poli the pufferfish,there goes polli the pufferfish,0.984
312,The Adventures of Lucy- The AI Dominance_chapt...,The Adventures of Lucy- The AI Dominance_chapt...,the adventures of lucy the ai dominance chapter 2,the adventures of lucy the ai dominance chapter 4,0.980
264,Detective Eve Dallas and The Stolen Legacy (PA...,Detective Eve Dallas and the Stolen Legacy (PA...,detective eve dallas and the stolen legacy part 1,detective eve dallas and the stolen legacy part 2,0.980
310,The Adventures of Lucy- The AI Dominance_chapt...,The Adventures of Lucy- The AI Dominance_chapt...,the adventures of lucy the ai dominance chapter 1,the adventures of lucy the ai dominance chapter 2,0.980
311,The Adventures of Lucy- The AI Dominance_chapt...,The Adventures of Lucy- The AI Dominance_chapt...,the adventures of lucy the ai dominance chapter 1,the adventures of lucy the ai dominance chapter 4,0.980
302,My Good Healthy Charter,My Good Health Charter,my good healthy charter,my good health charter,0.978
282,I Found Puppies - Book1,I Found Puppies - Book 1,i found puppies book1,i found puppies book 1,0.977
304,My Good Health Charter,MY GOOD HEALTH CHATER,my good health charter,my good health chater,0.977
259,Athena and Marina : Best Friends (Part 1),Athena and Marina: Best Friends (Part 2),athena and marina best friends part 1,athena and marina best friends part 2,0.973
0,A Book for Puchku,A Book for Puchkuu,a book for puchku,a book for puchkuu,0.971


In [12]:
# Check whether books with identical or highly similar titles also have identical or highly similar text
from itertools import combinations

def normalize_text_for_compare(text):
    return (
        str(text)
        .lower()
        .replace('\n', ' ')
        .strip()
    )

title_text_df = pd.DataFrame(v2_stories)[['unified_id', 'title', 'text', 'category_v1', 'reading_level_key']].copy()
title_text_df['normalized_title'] = (
    title_text_df['title']
    .fillna('')
    .str.lower()
    .str.replace(r'[^a-z0-9]+', ' ', regex=True)
    .str.strip()
)
title_text_df['normalized_text'] = title_text_df['text'].fillna('').apply(normalize_text_for_compare)

candidate_pairs = []
seen_pairs = set()

# 1. Pairs from exact duplicate titles
duplicate_groups = exact_duplicate_titles['normalized_title'].dropna().unique().tolist()
for normalized_title in duplicate_groups:
    group = title_text_df[title_text_df['normalized_title'] == normalized_title].reset_index(drop=True)
    for i, j in combinations(range(len(group)), 2):
        pair_key = tuple(sorted([group.loc[i, 'unified_id'], group.loc[j, 'unified_id']]))
        if pair_key not in seen_pairs:
            seen_pairs.add(pair_key)
            candidate_pairs.append((group.loc[i], group.loc[j], 'exact_title_match'))

# 2. Pairs from highly similar titles
if not similar_titles_df.empty:
    for _, row in similar_titles_df.iterrows():
        left_rows = title_text_df[title_text_df['title'] == row['title_1']]
        right_rows = title_text_df[title_text_df['title'] == row['title_2']]
        for _, left in left_rows.iterrows():
            for _, right in right_rows.iterrows():
                pair_key = tuple(sorted([left['unified_id'], right['unified_id']]))
                if left['unified_id'] != right['unified_id'] and pair_key not in seen_pairs:
                    seen_pairs.add(pair_key)
                    candidate_pairs.append((left, right, 'similar_title_match'))

comparison_rows = []
text_similarity_threshold = 0.95

for left, right, match_type in candidate_pairs:
    text_1 = left['normalized_text']
    text_2 = right['normalized_text']
    text_similarity = SequenceMatcher(None, text_1, text_2).ratio()
    comparison_rows.append({
        'match_type': match_type,
        'unified_id_1': left['unified_id'],
        'title_1': left['title'],
        'unified_id_2': right['unified_id'],
        'title_2': right['title'],
        'same_text': text_1 == text_2,
        'text_similarity': round(text_similarity, 3),
        'highly_similar_text': text_similarity >= text_similarity_threshold,
    })

title_text_compare_df = pd.DataFrame(comparison_rows).sort_values(
    ['same_text', 'text_similarity'],
    ascending=[False, False]
) if comparison_rows else pd.DataFrame()

print(f'Total candidate title pairs checked: {len(title_text_compare_df)}')
if not title_text_compare_df.empty:
    print(f'Pairs with identical text: {int(title_text_compare_df["same_text"].sum())}')
    print(f'Pairs with highly similar text (>= {text_similarity_threshold}): {int(title_text_compare_df["highly_similar_text"].sum())}')

display(title_text_compare_df.head(100))


Total candidate title pairs checked: 1462
Pairs with identical text: 7
Pairs with highly similar text (>= 0.95): 44


,match_type,unified_id_1,title_1,unified_id_2,title_2,same_text,text_similarity,highly_similar_text
110,exact_title_match,storyweaver_504581,Christmas,storyweaver_537887,Christmas,True,1.000,True
372,exact_title_match,storyweaver_406878,I love my family,storyweaver_409058,I love my family,True,1.000,True
401,exact_title_match,storyweaver_193089,Lion and the mouse,storyweaver_339769,Lion and the Mouse,True,1.000,True
414,exact_title_match,storyweaver_391712,Marching to Freedom,storyweaver_409998,Marching to Freedom,True,1.000,True
659,exact_title_match,storyweaver_167122,My little superhero,storyweaver_197191,My little superhero,True,1.000,True
...,...,...,...,...,...,...,...,...
1375,similar_title_match,storyweaver_659324,Alphabet - G,storyweaver_660455,Alphabet - L,False,0.399,False
143,exact_title_match,storyweaver_22926,Don't Wake the Baby!,storyweaver_277582,Don't Wake the Baby!,False,0.397,False
1209,similar_title_match,storyweaver_664051,Alphabet - T,storyweaver_664384,Alphabet - U,False,0.395,False
1342,similar_title_match,storyweaver_659616,Alphabet - I,storyweaver_660661,Alphabet - M,False,0.393,False


In [14]:
# Remove one book from each same-text duplicate group directly in V2, then recalculate category/level counts
from collections import defaultdict
import random

random_seed = 42
rng = random.Random(random_seed)

v2_path = LIBRARY_DIR / 'story_categories_v2.json'
delete_log_path = LIBRARY_DIR / 'story_categories_v2_deleted_same_text.json'

same_text_pairs = title_text_compare_df[title_text_compare_df['same_text'] == True].copy()

if same_text_pairs.empty:
    print('No same-text duplicate pairs found.')
else:
    v2_df = pd.DataFrame(v2_stories).copy()

    category_counts = v2_df.groupby('category_v1').size().to_dict()
    category_level_counts_map = v2_df.groupby(['category_v1', 'reading_level_key']).size().to_dict()

    adjacency = defaultdict(set)
    for _, row in same_text_pairs.iterrows():
        a = row['unified_id_1']
        b = row['unified_id_2']
        adjacency[a].add(b)
        adjacency[b].add(a)

    visited = set()
    components = []

    for node in adjacency:
        if node in visited:
            continue
        stack = [node]
        component = []
        while stack:
            current = stack.pop()
            if current in visited:
                continue
            visited.add(current)
            component.append(current)
            stack.extend(adjacency[current] - visited)
        if len(component) > 1:
            components.append(component)

    def rank_record(record):
        category = record['category_v1']
        level = record['reading_level_key']
        return (
            category_counts.get(category, 0),
            category_level_counts_map.get((category, level), 0),
            rng.random(),
        )

    decisions = []
    ids_to_delete = set()

    for component in components:
        component_df = v2_df[v2_df['unified_id'].isin(component)].copy()
        component_df['rank'] = component_df.apply(rank_record, axis=1)
        component_df = component_df.sort_values('rank').reset_index(drop=True)

        keep_row = component_df.iloc[0]
        delete_rows = component_df.iloc[1:]

        for _, delete_row in delete_rows.iterrows():
            ids_to_delete.add(delete_row['unified_id'])
            decisions.append({
                'keep_unified_id': keep_row['unified_id'],
                'keep_title': keep_row['title'],
                'keep_category_v1': keep_row['category_v1'],
                'keep_reading_level_key': keep_row['reading_level_key'],
                'keep_category_count': category_counts.get(keep_row['category_v1'], 0),
                'keep_category_level_count': category_level_counts_map.get((keep_row['category_v1'], keep_row['reading_level_key']), 0),
                'delete_unified_id': delete_row['unified_id'],
                'delete_title': delete_row['title'],
                'delete_category_v1': delete_row['category_v1'],
                'delete_reading_level_key': delete_row['reading_level_key'],
                'delete_category_count': category_counts.get(delete_row['category_v1'], 0),
                'delete_category_level_count': category_level_counts_map.get((delete_row['category_v1'], delete_row['reading_level_key']), 0),
                'component_size': len(component),
                'random_seed': random_seed,
            })

    deletion_decisions_df = pd.DataFrame(decisions)
    v2_df = v2_df[~v2_df['unified_id'].isin(ids_to_delete)].copy()

    with v2_path.open('w', encoding='utf-8') as f:
        json.dump(v2_df.to_dict(orient='records'), f, ensure_ascii=False, indent=2)

    with delete_log_path.open('w', encoding='utf-8') as f:
        json.dump(deletion_decisions_df.to_dict(orient='records'), f, ensure_ascii=False, indent=2)

    updated_counts = (
        v2_df
        .groupby(['category_v1', 'reading_level_key'])
        .size()
        .reset_index(name='book_count')
        .sort_values(['category_v1', 'reading_level_key'])
    )

    updated_pivot = (
        updated_counts
        .pivot(index='category_v1', columns='reading_level_key', values='book_count')
        .fillna(0)
        .astype(int)
    )

    print(f'Same-text duplicate groups: {len(components)}')
    print(f'Records deleted from V2: {len(ids_to_delete)}')
    print(f'Records remaining in V2: {len(v2_df)}')
    print(f'Overwrote cleaned V2 file at {v2_path}')
    print(f'Saved deletion log to {delete_log_path}')

    print('Updated V2 category/level counts:')

    display(updated_counts)
    display(updated_pivot)


Same-text duplicate groups: 7
Records deleted from V2: 7
Records remaining in V2: 7292
Overwrote cleaned V2 file at story_categories_v2.json
Saved deletion log to story_categories_v2_deleted_same_text.json
Updated V2 category/level counts:


,category_v1,reading_level_key,book_count
0,Animals,level_1,1009
1,Animals,level_2,1055
2,Animals,level_3,793
3,Daily Life,level_1,1154
4,Daily Life,level_2,1429
5,Daily Life,level_3,1430
6,Science & Knowledge,level_1,84
7,Science & Knowledge,level_2,117
8,Science & Knowledge,level_3,221


reading_level_key,level_1,level_2,level_3
category_v1,,,
Animals,1009,1055,793
Daily Life,1154,1429,1430
Science & Knowledge,84,117,221


In [15]:
# Check dirty / suspicious text patterns in V2
import re

v2_path = LIBRARY_DIR / 'story_categories_v2.json'

with v2_path.open(encoding='utf-8') as f:
    v2_stories = json.load(f)

dirty_df = pd.DataFrame(v2_stories).copy()
dirty_df['text'] = dirty_df['text'].fillna('').astype(str)

dirty_patterns = {
    'storyweaver_org': r'storyweaver\.org|storyweaver',
    'image_may_contain': r'image may contain',
    'page_marker': r'\bpage\s+\d+\b',
    'copyright': r'copyright|all rights reserved',
    'download_failed': r'download failed|failed to download',
    'html_tags': r'<[^>]+>',
    'ocr_garbage': r'[�]{2,}|[☐☒□■◆]{2,}',
    'url': r'https?://|www\.',
    'attribution': r'story attribution|images attributions?|original publisher|created by',
    'disclaimer': r'disclaimer:',
    'word_list': r'\bword list\b|glossary|activity:',
}

for name, pattern in dirty_patterns.items():
    dirty_df[name] = dirty_df['text'].str.contains(pattern, case=False, regex=True, na=False)

dirty_flag_cols = list(dirty_patterns.keys())
dirty_df['dirty_flag_count'] = dirty_df[dirty_flag_cols].sum(axis=1)
dirty_hits_df = dirty_df[dirty_df['dirty_flag_count'] > 0].copy()

print(f'Total V2 records checked: {len(dirty_df)}')
print(f'Records with at least one dirty-text flag: {len(dirty_hits_df)}')

dirty_summary = pd.DataFrame({
    'flag': dirty_flag_cols,
    'hit_count': [int(dirty_df[col].sum()) for col in dirty_flag_cols],
}).sort_values('hit_count', ascending=False)

display(dirty_summary)

dirty_hits_preview = dirty_hits_df[[
    'unified_id', 'title', 'category_v1', 'reading_level_key', 'word_count', 'dirty_flag_count', *dirty_flag_cols
]].sort_values(['dirty_flag_count', 'word_count'], ascending=[False, False])

display(dirty_hits_preview.head(100))

# Show a few text snippets for manual inspection
snippet_cols = ['unified_id', 'title', 'text']
dirty_snippets = dirty_hits_df[snippet_cols].copy()
dirty_snippets['text_snippet'] = dirty_snippets['text'].str.slice(0, 300)
display(dirty_snippets[['unified_id', 'title', 'text_snippet']].head(20))


Total V2 records checked: 7292
Records with at least one dirty-text flag: 257


,flag,hit_count
2,page_marker,153
7,url,48
0,storyweaver_org,27
8,attribution,24
10,word_list,11
3,copyright,8
5,html_tags,2
6,ocr_garbage,1
9,disclaimer,1
1,image_may_contain,0


,unified_id,title,category_v1,reading_level_key,word_count,dirty_flag_count,storyweaver_org,image_may_contain,page_marker,copyright,download_failed,html_tags,ocr_garbage,url,attribution,disclaimer,word_list
2203,storyweaver_111464,The Man in the Moon,Daily Life,level_3,1988,3,True,False,False,True,False,False,False,True,False,False,False
2306,storyweaver_122400,The Cat and the Fiddle,Daily Life,level_3,1806,3,True,False,False,True,False,False,False,True,False,False,False
2263,storyweaver_117384,Pussy-cat Mew,Animals,level_3,1252,3,True,False,False,True,False,False,False,True,False,False,False
2228,storyweaver_113407,How the Whale got his throat,Animals,level_3,1239,3,True,False,False,True,False,False,False,True,False,False,False
4905,storyweaver_433623,The Tumbling Words,Daily Life,level_3,1140,3,True,False,False,False,False,False,False,True,False,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1852,storyweaver_89408,Oil Spill!,Animals,level_3,722,1,False,False,True,False,False,False,False,False,False,False,False
1863,storyweaver_89807,The Missing Toy Cars,Daily Life,level_3,721,1,False,False,True,False,False,False,False,False,False,False,False
1815,storyweaver_88215,The Magical Blanket,Animals,level_3,719,1,False,False,True,False,False,False,False,False,False,False,False
1690,storyweaver_79779,Riding A Bicycle,Daily Life,level_3,717,1,False,False,True,False,False,False,False,False,False,False,False


,unified_id,title,text_snippet
65,storyweaver_1205,Pishi Caught in a Storm,Pishi was feeling sad and lonely. Just a day a...
66,storyweaver_1261,Thangwang and Bhalluka,Baby Thangwang likes to talk. GAA! GEE! GLUG G...
71,storyweaver_1591,Where is Gogo?,"He worked in a zoo. One day he saw that Gogo, ..."
93,storyweaver_2380,The Story of Stories,It was a beautiful summer day in the Forest-By...
103,storyweaver_2711,The Day It Rained Fish,Avanti was the zookeeper of Pitara zoo. There ...
154,storyweaver_4782,Bonda and Devi,"""Again! Stack them up again, Bonda! I want to ..."
187,storyweaver_5619,The Fascinating Fibonaccis,"Numbers. We use them every day. To count, meas..."
213,storyweaver_5990,Sharbani and the Elephant bird,In a beautiful village in the interiors of Ind...
226,storyweaver_6094,Tukkee and Her Parrot,Tukkee was a little girl who lived with her pa...
238,storyweaver_6242,"Chakora , The Brave Dog","Chakora, the stray dog with big eyes and black..."


In [17]:
# Minimally clean dirty text in V2 and overwrite V2 directly (with a backup)
import re
import shutil

v2_path = LIBRARY_DIR / 'story_categories_v2.json'
v2_backup_path = LIBRARY_DIR / 'story_categories_v2_backup_before_textclean.json'

with v2_path.open(encoding='utf-8') as f:
    v2_stories = json.load(f)

if not v2_backup_path.exists():
    shutil.copyfile(v2_path, v2_backup_path)

def clean_dirty_text(text):
    text = str(text)

    # Remove HTML tags first
    text = re.sub(r'<[^>]+>', ' ', text)

    # Remove obvious standalone junk lines / fragments
    text = re.sub(r'(?i)\bimage may contain:?\b', ' ', text)
    text = re.sub(r'(?i)\bstoryweaver\.org\b', ' ', text)
    text = re.sub(r'(?i)\bstoryweaver\b', ' ', text)
    text = re.sub(r'(?i)\bpage\s+\d+\b', ' ', text)
    text = re.sub(r'(?i)https?://\S+|www\.\S+', ' ', text)

    # Cut trailing metadata / publishing tails without touching the story body
    cut_markers = [
        'Story Attribution:',
        'Images Attributions:',
        'Image Attribution:',
        'Original Publisher',
        'Created by',
        'Disclaimer:',
        'Word List',
        'Glossary',
        'Activity:',
        'All rights reserved',
        'copyright',
        'Download failed',
        'Failed to download',
    ]

    lower_text = text.lower()
    cut_positions = []
    for marker in cut_markers:
        pos = lower_text.find(marker.lower())
        if pos != -1:
            cut_positions.append(pos)
    if cut_positions:
        text = text[:min(cut_positions)]

    # Remove repeated OCR-style junk symbols only
    text = re.sub(r'[�]{2,}|[☐☒□■◆]{2,}', ' ', text)

    # Normalize whitespace and punctuation spacing
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'\s+([,.;:!?])', r'\1', text)
    return text.strip()

cleaned_records = []
changed_count = 0
for record in v2_stories:
    updated = dict(record)
    old_text = str(updated.get('text', ''))
    new_text = clean_dirty_text(old_text)
    if new_text != old_text:
        changed_count += 1
    updated['text'] = new_text
    updated['word_count'] = len(new_text.split()) if new_text else 0
    cleaned_records.append(updated)

with v2_path.open('w', encoding='utf-8') as f:
    json.dump(cleaned_records, f, ensure_ascii=False, indent=2)

print(f'Total records processed: {len(cleaned_records)}')
print(f'Records whose text changed: {changed_count}')
print(f'Overwrote cleaned V2 file at {v2_path}')
print(f'Backup saved at {v2_backup_path}')

clean_compare_df = pd.DataFrame(cleaned_records)[['unified_id', 'title', 'word_count']].copy()
display(clean_compare_df.head(20))


Total records processed: 7292
Records whose text changed: 255
Overwrote cleaned V2 file at story_categories_v2.json
Backup saved at story_categories_v2_backup_before_textclean.json


,unified_id,title,word_count
0,storyweaver_21,Chuskit Goes to School!,1582
1,storyweaver_33,"Not Now, Not Now!",177
2,storyweaver_38,Counting on Moru,1960
3,storyweaver_54,Susheela's Kolams,210
4,storyweaver_67,"Sister, Sister, Where Does the Sun Go at Night?",1001
5,storyweaver_80,Aunty Jui's Baby,623
6,storyweaver_111,Little by Little,471
7,storyweaver_127,The Missing Bat,555
8,storyweaver_132,Daddy's Mo,416
9,storyweaver_141,Going to a Wedding,285


In [18]:
# Check for low-information / highly repetitive text in V2 using unique word ratio
v2_path = LIBRARY_DIR / 'story_categories_v2.json'

with v2_path.open(encoding='utf-8') as f:
    v2_stories = json.load(f)

repeat_df = pd.DataFrame(v2_stories).copy()
repeat_df['text'] = repeat_df['text'].fillna('').astype(str)

def unique_ratio(text):
    words = text.lower().split()
    if len(words) == 0:
        return 0
    return len(set(words)) / len(words)

repeat_df['unique_ratio'] = repeat_df['text'].apply(unique_ratio)

# 只重点看词数已经不短、但 unique ratio 很低的文本
suspect_repeat_df = repeat_df[(repeat_df['word_count'] >= 30) & (repeat_df['unique_ratio'] < 0.3)].copy()
suspect_repeat_df = suspect_repeat_df.sort_values(['unique_ratio', 'word_count'])

print(f'Total V2 records checked: {len(repeat_df)}')
print(f'Suspected repetitive-text records: {len(suspect_repeat_df)}')

display(suspect_repeat_df[['unified_id', 'title', 'category_v1', 'reading_level_key', 'word_count', 'unique_ratio']].head(100))

repeat_snippets = suspect_repeat_df[['unified_id', 'title', 'text']].copy()
repeat_snippets['text_snippet'] = repeat_snippets['text'].str.slice(0, 300)
display(repeat_snippets[['unified_id', 'title', 'text_snippet']].head(20))


Total V2 records checked: 7292
Suspected repetitive-text records: 29


,unified_id,title,category_v1,reading_level_key,word_count,unique_ratio
5417,storyweaver_487389,A wise deer and Crowdly Tiger,Animals,level_2,1790,0.135196
6043,storyweaver_578926,Giant beast vs kaiju,Animals,level_1,1875,0.152533
6571,storyweaver_628701,About Raksha bhandan,Daily Life,level_2,590,0.161017
6901,storyweaver_661028,sitree,Science & Knowledge,level_2,233,0.163090
3120,storyweaver_193790,building up the temple : poem,Daily Life,level_1,104,0.163462
736,storyweaver_21894,One Beautiful Tree,Science & Knowledge,level_2,249,0.180723
4696,storyweaver_400593,Tuni teaches the 'Doctor' a lesson,Animals,level_2,1031,0.183317
5097,storyweaver_459373,Good to read,Animals,level_1,92,0.184783
812,storyweaver_23993,A shopping trip to remember!,Daily Life,level_2,1750,0.197714
1410,storyweaver_53451,The world has no rainbow,Science & Knowledge,level_2,125,0.200000


,unified_id,title,text_snippet
5417,storyweaver_487389,A wise deer and Crowdly Tiger,This Short Story A Wise Deer and A Cowardly Ti...
6043,storyweaver_578926,Giant beast vs kaiju,Lions were hunting the wildebeest for there me...
6571,storyweaver_628701,About Raksha bhandan,I will make my sister a software engineer beca...
6901,storyweaver_661028,sitree,This is the Tree. Trees grow on the ground. Th...
3120,storyweaver_193790,building up the temple : poem,"building up the temple, building up the temple..."
736,storyweaver_21894,One Beautiful Tree,It is standing on the ground It is a green gre...
4696,storyweaver_400593,Tuni teaches the 'Doctor' a lesson,Once there lived teeny-weeny bird named 'Tuni'...
5097,storyweaver_459373,Good to read,﻿ புலி ஒன்று பசியோடு தன் உணவைத் தேடி கொண்டு இர...
812,storyweaver_23993,A shopping trip to remember!,Jelly and I were heading toward the local mall...
1410,storyweaver_53451,The world has no rainbow,"The world is a duck bro, we have a Trump bro, ..."


In [22]:
# Tag only the currently detected suspicious repetitive-text records for analysis (do not modify V2)
import re

analysis_df = suspect_repeat_df.copy()

def classify_text_quality(row):
    text = str(row['text']).lower()
    title = str(row['title']).lower()
    unique_ratio = row['unique_ratio']

    if re.search(r'copyright|all rights reserved|story attribution|images attributions?|disclaimer:|glossary|activity:|original publisher|created by', text):
        return 'metadata_noise'

    if re.search(r'poem|rhyme|song|chant|nursery', title) or re.search(r'poem|rhyme|song|chant|nursery', text):
        return 'repetitive_poem'

    if unique_ratio < 0.15:
        return 'suspicious_low_information'

    if unique_ratio < 0.30:
        return 'repetitive_style'

    return 'normal'

analysis_df['text_quality_flag'] = analysis_df.apply(classify_text_quality, axis=1)

flag_summary = (
    analysis_df['text_quality_flag']
    .value_counts()
    .rename_axis('text_quality_flag')
    .reset_index(name='book_count')
)

display(flag_summary)

analysis_preview = analysis_df[[
    'unified_id', 'title', 'category_v1', 'reading_level_key', 'word_count', 'unique_ratio', 'text_quality_flag'
]].sort_values(['text_quality_flag', 'unique_ratio', 'word_count'])

display(analysis_preview)

analysis_snippets = analysis_df[['unified_id', 'title', 'text_quality_flag', 'text']].copy()
analysis_snippets['text_snippet'] = analysis_snippets['text'].str.slice(0, 300)
display(analysis_snippets[['unified_id', 'title', 'text_quality_flag', 'text_snippet']])


,text_quality_flag,book_count
0,repetitive_style,24
1,repetitive_poem,4
2,suspicious_low_information,1


,unified_id,title,category_v1,reading_level_key,word_count,unique_ratio,text_quality_flag
3120,storyweaver_193790,building up the temple : poem,Daily Life,level_1,104,0.163462,repetitive_poem
5021,storyweaver_450363,Fly-Fall Poems by B.K.Tirumalamma,Science & Knowledge,level_1,590,0.272881,repetitive_poem
83,storyweaver_2057,The Rainbow Story,Animals,level_2,753,0.280212,repetitive_poem
618,storyweaver_16939,What is it?,Animals,level_1,104,0.298077,repetitive_poem
6043,storyweaver_578926,Giant beast vs kaiju,Animals,level_1,1875,0.152533,repetitive_style
6571,storyweaver_628701,About Raksha bhandan,Daily Life,level_2,590,0.161017,repetitive_style
6901,storyweaver_661028,sitree,Science & Knowledge,level_2,233,0.163090,repetitive_style
736,storyweaver_21894,One Beautiful Tree,Science & Knowledge,level_2,249,0.180723,repetitive_style
4696,storyweaver_400593,Tuni teaches the 'Doctor' a lesson,Animals,level_2,1031,0.183317,repetitive_style
5097,storyweaver_459373,Good to read,Animals,level_1,92,0.184783,repetitive_style


,unified_id,title,text_quality_flag,text_snippet
5417,storyweaver_487389,A wise deer and Crowdly Tiger,suspicious_low_information,This Short Story A Wise Deer and A Cowardly Ti...
6043,storyweaver_578926,Giant beast vs kaiju,repetitive_style,Lions were hunting the wildebeest for there me...
6571,storyweaver_628701,About Raksha bhandan,repetitive_style,I will make my sister a software engineer beca...
6901,storyweaver_661028,sitree,repetitive_style,This is the Tree. Trees grow on the ground. Th...
3120,storyweaver_193790,building up the temple : poem,repetitive_poem,"building up the temple, building up the temple..."
736,storyweaver_21894,One Beautiful Tree,repetitive_style,It is standing on the ground It is a green gre...
4696,storyweaver_400593,Tuni teaches the 'Doctor' a lesson,repetitive_style,Once there lived teeny-weeny bird named 'Tuni'...
5097,storyweaver_459373,Good to read,repetitive_style,﻿ புலி ஒன்று பசியோடு தன் உணவைத் தேடி கொண்டு இர...
812,storyweaver_23993,A shopping trip to remember!,repetitive_style,Jelly and I were heading toward the local mall...
1410,storyweaver_53451,The world has no rainbow,repetitive_style,"The world is a duck bro, we have a Trump bro, ..."


In [23]:
# Analyze which flagged repetitive-text records should be deleted, then delete them from V2
v2_path = LIBRARY_DIR / 'story_categories_v2.json'
delete_log_path = LIBRARY_DIR / 'story_categories_v2_deleted_low_information.json'

# Conservative delete rule:
# - delete: metadata_noise, suspicious_low_information
# - keep for now: repetitive_poem, repetitive_style, normal
delete_flags = {'metadata_noise', 'suspicious_low_information'}

delete_candidates_df = analysis_df[analysis_df['text_quality_flag'].isin(delete_flags)].copy()
keep_candidates_df = analysis_df[~analysis_df['text_quality_flag'].isin(delete_flags)].copy()

print('Recommended delete flags:', sorted(delete_flags))
print(f'Records recommended for deletion: {len(delete_candidates_df)}')
print(f'Records kept for now: {len(keep_candidates_df)}')

display(delete_candidates_df[['unified_id', 'title', 'category_v1', 'reading_level_key', 'word_count', 'unique_ratio', 'text_quality_flag']].sort_values(['text_quality_flag', 'unique_ratio', 'word_count']))

with v2_path.open(encoding='utf-8') as f:
    current_v2_stories = json.load(f)

current_v2_df = pd.DataFrame(current_v2_stories)
delete_ids = set(delete_candidates_df['unified_id'])
updated_v2_df = current_v2_df[~current_v2_df['unified_id'].isin(delete_ids)].copy()

with v2_path.open('w', encoding='utf-8') as f:
    json.dump(updated_v2_df.to_dict(orient='records'), f, ensure_ascii=False, indent=2)

with delete_log_path.open('w', encoding='utf-8') as f:
    json.dump(delete_candidates_df.to_dict(orient='records'), f, ensure_ascii=False, indent=2)

updated_counts = (
    updated_v2_df
    .groupby(['category_v1', 'reading_level_key'])
    .size()
    .reset_index(name='book_count')
    .sort_values(['category_v1', 'reading_level_key'])
)

updated_pivot = (
    updated_counts
    .pivot(index='category_v1', columns='reading_level_key', values='book_count')
    .fillna(0)
    .astype(int)
)

print(f'Records deleted from V2: {len(delete_ids)}')
print(f'Records remaining in V2: {len(updated_v2_df)}')
print(f'Overwrote V2 at {v2_path}')
print(f'Saved delete log to {delete_log_path}')

print('Updated V2 counts by category_v2 and reading_level_key:')

display(updated_counts)
display(updated_pivot)


Recommended delete flags: ['metadata_noise', 'suspicious_low_information']
Records recommended for deletion: 1
Records kept for now: 28


,unified_id,title,category_v1,reading_level_key,word_count,unique_ratio,text_quality_flag
5417,storyweaver_487389,A wise deer and Crowdly Tiger,Animals,level_2,1790,0.135196,suspicious_low_information


Records deleted from V2: 1
Records remaining in V2: 7291
Overwrote V2 at story_categories_v2.json
Saved delete log to story_categories_v2_deleted_low_information.json
Updated V2 counts by category_v2 and reading_level_key:


,category_v1,reading_level_key,book_count
0,Animals,level_1,1009
1,Animals,level_2,1054
2,Animals,level_3,793
3,Daily Life,level_1,1154
4,Daily Life,level_2,1429
5,Daily Life,level_3,1430
6,Science & Knowledge,level_1,84
7,Science & Knowledge,level_2,117
8,Science & Knowledge,level_3,221


reading_level_key,level_1,level_2,level_3
category_v1,,,
Animals,1009,1054,793
Daily Life,1154,1429,1430
Science & Knowledge,84,117,221


In [24]:
# Check whether any book in V2 contains non-English content in title or text
import re

v2_path = LIBRARY_DIR / 'story_categories_v2.json'

with v2_path.open(encoding='utf-8') as f:
    v2_stories = json.load(f)

lang_df = pd.DataFrame(v2_stories).copy()
lang_df['title'] = lang_df['title'].fillna('').astype(str)
lang_df['text'] = lang_df['text'].fillna('').astype(str)

# Detect obvious non-Latin / non-English script characters
non_english_char_pattern = re.compile(r'[\u0080-\u024F\u0370-\u03FF\u0400-\u04FF\u0590-\u05FF\u0600-\u06FF\u0900-\u097F\u0980-\u09FF\u0A00-\u0A7F\u0A80-\u0AFF\u0B00-\u0B7F\u0B80-\u0BFF\u0C00-\u0C7F\u0C80-\u0CFF\u0D00-\u0D7F\u0E00-\u0E7F\u4E00-\u9FFF]')

lang_df['title_has_non_english'] = lang_df['title'].apply(lambda x: bool(non_english_char_pattern.search(x)))
lang_df['text_has_non_english'] = lang_df['text'].apply(lambda x: bool(non_english_char_pattern.search(x)))
lang_df['has_non_english'] = lang_df['title_has_non_english'] | lang_df['text_has_non_english']

non_english_df = lang_df[lang_df['has_non_english']].copy()

print(f'Total V2 records checked: {len(lang_df)}')
print(f'Records with non-English content in title or text: {len(non_english_df)}')

non_english_summary = pd.DataFrame({
    'flag': ['title_has_non_english', 'text_has_non_english', 'has_non_english'],
    'book_count': [
        int(lang_df['title_has_non_english'].sum()),
        int(lang_df['text_has_non_english'].sum()),
        int(lang_df['has_non_english'].sum()),
    ]
})

display(non_english_summary)

display(non_english_df[['unified_id', 'title', 'category_v1', 'reading_level_key', 'title_has_non_english', 'text_has_non_english']].head(100))

non_english_snippets = non_english_df[['unified_id', 'title', 'text']].copy()
non_english_snippets['text_snippet'] = non_english_snippets['text'].str.slice(0, 300)
display(non_english_snippets[['unified_id', 'title', 'text_snippet']].head(20))


Total V2 records checked: 7291
Records with non-English content in title or text: 240


,flag,book_count
0,title_has_non_english,3
1,text_has_non_english,240
2,has_non_english,240


,unified_id,title,category_v1,reading_level_key,title_has_non_english,text_has_non_english
10,storyweaver_144,Going Home,Daily Life,level_2,False,True
301,storyweaver_8003,How do aeroplanes fly,Science & Knowledge,level_2,False,True
469,storyweaver_12513,The Dance of the Flamingo,Animals,level_3,False,True
493,storyweaver_13302,Razia returns to Chennai,Daily Life,level_2,False,True
519,storyweaver_13834,In the pursuit of dragonflies,Animals,level_1,False,True
...,...,...,...,...,...,...
3470,storyweaver_232667,Safar Nilu ka,Daily Life,level_3,False,True
3471,storyweaver_232802,Dans,Daily Life,level_2,False,True
3492,storyweaver_236502,Goals,Daily Life,level_3,False,True
3497,storyweaver_237002,Our Earth,Daily Life,level_3,False,True


,unified_id,title,text_snippet
10,storyweaver_144,Going Home,School is over. Children are leaving. Teachers...
301,storyweaver_8003,How do aeroplanes fly,"One day, during science class, she looked out ..."
469,storyweaver_12513,The Dance of the Flamingo,"Like ballet dancers they pirouette on one leg,..."
493,storyweaver_13302,Razia returns to Chennai,Razia with her grandparents is excited to visi...
519,storyweaver_13834,In the pursuit of dragonflies,"One of them said to the others, ""This reminds ..."
579,storyweaver_15165,The lucky ticket,"Once upon a time one kid named Margarita, she ..."
587,storyweaver_15219,Baby Dinasour,Once upon a time there was a baby dinasour tha...
645,storyweaver_18256,Round,This bus does not move. ¿Why are the bracelets...
810,storyweaver_23622,May Searches for the Sea,He teaches us the alphabet and numbers. None o...
812,storyweaver_23993,A shopping trip to remember!,Jelly and I were heading toward the local mall...


In [25]:
# Delete non-English records from V2 and recalculate category/level counts
v2_path = LIBRARY_DIR / 'story_categories_v2.json'
delete_log_path = LIBRARY_DIR / 'story_categories_v2_deleted_non_english.json'

delete_non_english_df = non_english_df.copy()
delete_ids = set(delete_non_english_df['unified_id'])

print(f'Records to delete for non-English content: {len(delete_ids)}')
display(delete_non_english_df[['unified_id', 'title', 'category_v1', 'reading_level_key', 'title_has_non_english', 'text_has_non_english']].head(100))

with v2_path.open(encoding='utf-8') as f:
    current_v2_stories = json.load(f)

current_v2_df = pd.DataFrame(current_v2_stories)
updated_v2_df = current_v2_df[~current_v2_df['unified_id'].isin(delete_ids)].copy()

with v2_path.open('w', encoding='utf-8') as f:
    json.dump(updated_v2_df.to_dict(orient='records'), f, ensure_ascii=False, indent=2)

with delete_log_path.open('w', encoding='utf-8') as f:
    json.dump(delete_non_english_df.to_dict(orient='records'), f, ensure_ascii=False, indent=2)

updated_counts = (
    updated_v2_df
    .groupby(['category_v1', 'reading_level_key'])
    .size()
    .reset_index(name='book_count')
    .sort_values(['category_v1', 'reading_level_key'])
)

updated_pivot = (
    updated_counts
    .pivot(index='category_v1', columns='reading_level_key', values='book_count')
    .fillna(0)
    .astype(int)
)

print(f'Records deleted from V2: {len(delete_ids)}')
print(f'Records remaining in V2: {len(updated_v2_df)}')
print(f'Overwrote V2 at {v2_path}')
print(f'Saved deletion log to {delete_log_path}')
print('Updated V2 counts by category_v1 and reading_level_key:')

display(updated_counts)
display(updated_pivot)


Records to delete for non-English content: 240


,unified_id,title,category_v1,reading_level_key,title_has_non_english,text_has_non_english
10,storyweaver_144,Going Home,Daily Life,level_2,False,True
301,storyweaver_8003,How do aeroplanes fly,Science & Knowledge,level_2,False,True
469,storyweaver_12513,The Dance of the Flamingo,Animals,level_3,False,True
493,storyweaver_13302,Razia returns to Chennai,Daily Life,level_2,False,True
519,storyweaver_13834,In the pursuit of dragonflies,Animals,level_1,False,True
...,...,...,...,...,...,...
3470,storyweaver_232667,Safar Nilu ka,Daily Life,level_3,False,True
3471,storyweaver_232802,Dans,Daily Life,level_2,False,True
3492,storyweaver_236502,Goals,Daily Life,level_3,False,True
3497,storyweaver_237002,Our Earth,Daily Life,level_3,False,True


Records deleted from V2: 240
Records remaining in V2: 7051
Overwrote V2 at story_categories_v2.json
Saved deletion log to story_categories_v2_deleted_non_english.json
Updated V2 counts by category_v1 and reading_level_key:


,category_v1,reading_level_key,book_count
0,Animals,level_1,980
1,Animals,level_2,1026
2,Animals,level_3,771
3,Daily Life,level_1,1123
4,Daily Life,level_2,1388
5,Daily Life,level_3,1361
6,Science & Knowledge,level_1,81
7,Science & Knowledge,level_2,112
8,Science & Knowledge,level_3,209


reading_level_key,level_1,level_2,level_3
category_v1,,,
Animals,980,1026,771
Daily Life,1123,1388,1361
Science & Knowledge,81,112,209


In [26]:
# Check whether V2 contains unusually uppercase text (possible OCR issue)
v2_path = LIBRARY_DIR / 'story_categories_v2.json'

with v2_path.open(encoding='utf-8') as f:
    v2_stories = json.load(f)

upper_df = pd.DataFrame(v2_stories).copy()
upper_df['text'] = upper_df['text'].fillna('').astype(str)

def uppercase_ratio(text):
    letters = [c for c in text if c.isalpha()]
    if not letters:
        return 0
    upper = sum(c.isupper() for c in letters)
    return upper / len(letters)

upper_df['upper_ratio'] = upper_df['text'].apply(uppercase_ratio)

# Focus on records whose main text is mostly uppercase
suspect_upper_df = upper_df[(upper_df['word_count'] >= 20) & (upper_df['upper_ratio'] >= 0.5)].copy()
suspect_upper_df = suspect_upper_df.sort_values(['upper_ratio', 'word_count'], ascending=[False, False])

print(f'Total V2 records checked: {len(upper_df)}')
print(f'Suspected uppercase-heavy records: {len(suspect_upper_df)}')

display(suspect_upper_df[['unified_id', 'title', 'category_v1', 'reading_level_key', 'word_count', 'upper_ratio']].head(100))

upper_snippets = suspect_upper_df[['unified_id', 'title', 'text']].copy()
upper_snippets['text_snippet'] = upper_snippets['text'].str.slice(0, 300)
display(upper_snippets[['unified_id', 'title', 'text_snippet']].head(20))


Total V2 records checked: 7051
Suspected uppercase-heavy records: 106


,unified_id,title,category_v1,reading_level_key,word_count,upper_ratio
5631,storyweaver_534839,LEGENDARY STORY OF MONKEY AND HIS LAZY FRIENDS,Animals,level_2,681,1.000000
5270,storyweaver_489022,FORESTER,Daily Life,level_1,470,1.000000
4716,storyweaver_427119,ADIVASIS 4,Daily Life,level_3,442,1.000000
2180,storyweaver_114101,CHINNU NEGGINDI,Daily Life,level_2,372,1.000000
5437,storyweaver_511988,A BUSY MARKET AND A BUSY DAY .,Daily Life,level_3,357,1.000000
...,...,...,...,...,...,...
4495,storyweaver_388805,BHUJI AND SONA'S FAVOURITE FOOD,Daily Life,level_2,61,0.856574
4151,storyweaver_335970,CRAZE IN ANIMALS,Animals,level_2,184,0.785057
4507,storyweaver_391126,PENGUINS FROM MADAGASCAR 1,Animals,level_3,106,0.776256
2827,storyweaver_171044,SOAPY WATER,Daily Life,level_2,120,0.772824


,unified_id,title,text_snippet
5631,storyweaver_534839,LEGENDARY STORY OF MONKEY AND HIS LAZY FRIENDS,"THERE WAS A TIME IN ANIMALIA, AN ANIMAL KINGDO..."
5270,storyweaver_489022,FORESTER,ONCE A FORESTER WHO WENT INTO THE FOREST TO HU...
4716,storyweaver_427119,ADIVASIS 4,WHEN HER MOTHER CAME TO TAKE HER HOME. SHE WEN...
2180,storyweaver_114101,CHINNU NEGGINDI,I EE AADHIVARAM PODDUNA KARATE PARIKSHA UNDI T...
5437,storyweaver_511988,A BUSY MARKET AND A BUSY DAY .,WHEN MY MOTHER GIVES ME SOME MONEY AND TELLS M...
5274,storyweaver_489360,THE PRECIOUS GIFT,ONCE UPON A TIME THERE WAS A CUTE LITTLE GIRL ...
5332,storyweaver_496372,the rabbit's clever,THE RABBIT'S CLEVER ONCE A UPON A TIME THERE W...
2515,storyweaver_144607,THE EVIL CAKE,ONE DAY IT WAS MIA'S BIRTH DAY.SHE WENT TO A C...
2480,storyweaver_141039,CHRISTMAS PARTY,IT WAS WINTER. SWEETY WAS BUILDING MANY SNOW M...
4697,storyweaver_424137,JOURNEY OF LIFE.,I DO. CAN I HELP ANYONE PACKING FOR SOMETHING....


In [27]:
# Normalize uppercase-heavy text in V2 for records with upper_ratio >= 0.7
v2_path = LIBRARY_DIR / 'story_categories_v2.json'
upper_log_path = LIBRARY_DIR / 'story_categories_v2_uppercase_normalized.json'

with v2_path.open(encoding='utf-8') as f:
    v2_stories = json.load(f)

v2_upper_df = pd.DataFrame(v2_stories).copy()
v2_upper_df['text'] = v2_upper_df['text'].fillna('').astype(str)

def uppercase_ratio(text):
    letters = [c for c in text if c.isalpha()]
    if not letters:
        return 0
    upper = sum(c.isupper() for c in letters)
    return upper / len(letters)

def normalize_uppercase_text(text):
    text = str(text).strip()
    text = text.lower()
    # Capitalize sentence starts conservatively
    text = re.sub(r'(^|[.!?]\s+)([a-z])', lambda m: m.group(1) + m.group(2).upper(), text)
    return text

v2_upper_df['upper_ratio'] = v2_upper_df['text'].apply(uppercase_ratio)
suspect_upper_df = v2_upper_df[(v2_upper_df['word_count'] >= 20) & (v2_upper_df['upper_ratio'] >= 0.5)].copy()
to_normalize_df = suspect_upper_df[suspect_upper_df['upper_ratio'] >= 0.7].copy()
manual_check_df = suspect_upper_df[suspect_upper_df['upper_ratio'] < 0.7].copy()

normalize_ids = set(to_normalize_df['unified_id'])
manual_check_ids = manual_check_df['unified_id'].tolist()

changed_rows = []
updated_records = []
for record in v2_stories:
    updated = dict(record)
    if updated['unified_id'] in normalize_ids:
        old_text = str(updated.get('text', ''))
        new_text = normalize_uppercase_text(old_text)
        updated['text'] = new_text
        updated['word_count'] = len(new_text.split()) if new_text else 0
        changed_rows.append({
            'unified_id': updated['unified_id'],
            'title': updated.get('title', ''),
            'old_upper_ratio': float(to_normalize_df.loc[to_normalize_df['unified_id'] == updated['unified_id'], 'upper_ratio'].iloc[0]),
        })
    updated_records.append(updated)

with v2_path.open('w', encoding='utf-8') as f:
    json.dump(updated_records, f, ensure_ascii=False, indent=2)

with upper_log_path.open('w', encoding='utf-8') as f:
    json.dump(changed_rows, f, ensure_ascii=False, indent=2)

print(f'Records normalized in V2: {len(changed_rows)}')
print(f'Manual-check records (< 0.7): {len(manual_check_ids)}')
print('Manual-check unified_ids:')
for uid in manual_check_ids:
    print(uid)

display(manual_check_df[['unified_id', 'title', 'word_count', 'upper_ratio']].sort_values('upper_ratio'))


Records normalized in V2: 100
Manual-check records (< 0.7): 6
Manual-check unified_ids:
storyweaver_393751
storyweaver_396170
storyweaver_426294
storyweaver_456125
storyweaver_570877
storyweaver_639001


,unified_id,title,word_count,upper_ratio
4521,storyweaver_393751,HIDE AND SEEK!!!!!,110,0.585127
4712,storyweaver_426294,CRICKET IS MY FIRST OVER,947,0.598475
6442,storyweaver_639001,THE MONKEY AND THE CROCODILE,244,0.614719
4537,storyweaver_396170,FROG LIFE CYCLE,126,0.615238
5817,storyweaver_570877,Ramya with her cat,185,0.644388
4922,storyweaver_456125,A gift for a friend,142,0.683112


In [28]:
# Apply manual decisions for the remaining 6 uppercase-heavy books in V2
v2_path = LIBRARY_DIR / 'story_categories_v2.json'
manual_log_path = LIBRARY_DIR / 'story_categories_v2_manual_uppercase_actions.json'

manual_actions = {
    'HIDE AND SEEK!!!!!': 'keep_normalize',
    'FROG LIFE CYCLE': 'keep_normalize',
    'CRICKET IS MY FIRST OVER': 'delete',
    'A gift for a friend': 'keep_normalize',
    'Ramya with her cat': 'delete',
    'THE MONKEY AND THE CROCODILE': 'keep_normalize',
}

with v2_path.open(encoding='utf-8') as f:
    current_v2_stories = json.load(f)

def normalize_uppercase_text(text):
    text = str(text).strip()
    text = text.lower()
    text = re.sub(r'(^|[.!?]\s+)([a-z])', lambda m: m.group(1) + m.group(2).upper(), text)
    return text

updated_records = []
action_log = []
deleted_count = 0
normalized_count = 0

for record in current_v2_stories:
    title = str(record.get('title', ''))
    action = manual_actions.get(title)

    if action == 'delete':
        deleted_count += 1
        action_log.append({
            'unified_id': record.get('unified_id'),
            'title': title,
            'action': 'delete',
        })
        continue

    updated = dict(record)
    if action == 'keep_normalize':
        old_text = str(updated.get('text', ''))
        new_text = normalize_uppercase_text(old_text)
        updated['text'] = new_text
        updated['word_count'] = len(new_text.split()) if new_text else 0
        normalized_count += 1
        action_log.append({
            'unified_id': updated.get('unified_id'),
            'title': title,
            'action': 'keep_normalize',
        })

    updated_records.append(updated)

with v2_path.open('w', encoding='utf-8') as f:
    json.dump(updated_records, f, ensure_ascii=False, indent=2)

with manual_log_path.open('w', encoding='utf-8') as f:
    json.dump(action_log, f, ensure_ascii=False, indent=2)

updated_v2_df = pd.DataFrame(updated_records)
updated_counts = (
    updated_v2_df
    .groupby(['category_v1', 'reading_level_key'])
    .size()
    .reset_index(name='book_count')
    .sort_values(['category_v1', 'reading_level_key'])
)

updated_pivot = (
    updated_counts
    .pivot(index='category_v1', columns='reading_level_key', values='book_count')
    .fillna(0)
    .astype(int)
)

print(f'Records normalized: {normalized_count}')
print(f'Records deleted: {deleted_count}')
print(f'Records remaining in V2: {len(updated_records)}')
print(f'Overwrote V2 at {v2_path}')
print(f'Saved manual action log to {manual_log_path}')
display(pd.DataFrame(action_log))
display(updated_counts)
display(updated_pivot)


Records normalized: 5
Records deleted: 2
Records remaining in V2: 7049
Overwrote V2 at story_categories_v2.json
Saved manual action log to story_categories_v2_manual_uppercase_actions.json


,unified_id,title,action
0,storyweaver_393751,HIDE AND SEEK!!!!!,keep_normalize
1,storyweaver_396170,FROG LIFE CYCLE,keep_normalize
2,storyweaver_426294,CRICKET IS MY FIRST OVER,delete
3,storyweaver_456125,A gift for a friend,keep_normalize
4,storyweaver_570877,Ramya with her cat,delete
5,storyweaver_628010,THE MONKEY AND THE CROCODILE,keep_normalize
6,storyweaver_639001,THE MONKEY AND THE CROCODILE,keep_normalize


,category_v1,reading_level_key,book_count
0,Animals,level_1,979
1,Animals,level_2,1026
2,Animals,level_3,771
3,Daily Life,level_1,1123
4,Daily Life,level_2,1388
5,Daily Life,level_3,1360
6,Science & Knowledge,level_1,81
7,Science & Knowledge,level_2,112
8,Science & Knowledge,level_3,209


reading_level_key,level_1,level_2,level_3
category_v1,,,
Animals,979,1026,771
Daily Life,1123,1388,1360
Science & Knowledge,81,112,209


In [29]:
# Check whether V2 contains texts with too many special characters
v2_path = LIBRARY_DIR / 'story_categories_v2.json'

with v2_path.open(encoding='utf-8') as f:
    v2_stories = json.load(f)

special_df = pd.DataFrame(v2_stories).copy()
special_df['text'] = special_df['text'].fillna('').astype(str)

def special_char_ratio(text):
    special = sum((not c.isalnum()) and (not c.isspace()) for c in text)
    return special / max(len(text), 1)

special_df['special_ratio'] = special_df['text'].apply(special_char_ratio)

# Focus on records with relatively high special-character ratio
suspect_special_df = special_df[(special_df['word_count'] >= 20) & (special_df['special_ratio'] >= 0.1)].copy()
suspect_special_df = suspect_special_df.sort_values(['special_ratio', 'word_count'], ascending=[False, False])

print(f'Total V2 records checked: {len(special_df)}')
print(f'Suspected special-character-heavy records: {len(suspect_special_df)}')

display(suspect_special_df[['unified_id', 'title', 'category_v1', 'reading_level_key', 'word_count', 'special_ratio']].head(100))

special_snippets = suspect_special_df[['unified_id', 'title', 'text']].copy()
special_snippets['text_snippet'] = special_snippets['text'].str.slice(0, 300)
display(special_snippets[['unified_id', 'title', 'text_snippet']].head(20))


Total V2 records checked: 7049
Suspected special-character-heavy records: 87


,unified_id,title,category_v1,reading_level_key,word_count,special_ratio
6962,storyweaver_687170,Ammu's Sunny Discovery,Daily Life,level_2,363,0.497862
6032,storyweaver_603270,Deep Jungle,Animals,level_1,98,0.493802
6406,storyweaver_634986,the first olympiad exam,Science & Knowledge,level_3,180,0.471500
6054,storyweaver_605976,One Wonderful Vacation,Daily Life,level_1,142,0.335345
3592,storyweaver_266769,Little Miss Ant,Animals,level_1,86,0.297659
...,...,...,...,...,...,...
1454,storyweaver_61023,I Don't Want to go to Sleep!,Daily Life,level_2,218,0.100671
3039,storyweaver_193800,I can be...anything!,Animals,level_2,101,0.100642
1364,storyweaver_52153,Kittu's new shoes,Daily Life,level_1,74,0.100264
455,storyweaver_12275,April fool.... have fun!,Animals,level_2,242,0.100075


,unified_id,title,text_snippet
6962,storyweaver_687170,Ammu's Sunny Discovery,Ammu has a yellow ball. She plays with it ever...
6032,storyweaver_603270,Deep Jungle,Ha Ha Haaaa. The king of the jungle is jumping...
6406,storyweaver_634986,the first olympiad exam,"The Olympiads, IMO (International Maths Olympi..."
6054,storyweaver_605976,One Wonderful Vacation,"""YEAY!!! it's almost time!"" yells Mittu. Mittu..."
3592,storyweaver_266769,Little Miss Ant,One fine day she walked real slow to where a m...
3344,storyweaver_229847,Quiz - Count How Many?,"In the pictures shown in the book, find out th..."
762,storyweaver_22557,Mitu's friends,Mitu was a sweet girl. She lived in a farm hou...
2400,storyweaver_136564,Say in a breath,"the grass, say in a breath. Fur...Fur....Fur....."
1365,storyweaver_52198,nina part 2,﻿ ﻿ ﻿ ﻿ j ﻿ ﻿ ﻿ ﻿ ﻿Important letter from the a...
1824,storyweaver_90549,JOKER,My name is Rekoj. I live in POTATO LAND. I lov...


In [30]:
# Process special-character-heavy texts in V2 based on special_ratio rules
v2_path = LIBRARY_DIR / 'story_categories_v2.json'
normalize_log_path = LIBRARY_DIR / 'story_categories_v2_special_normalized.json'
delete_log_path = LIBRARY_DIR / 'story_categories_v2_deleted_corrupted_special.json'

with v2_path.open(encoding='utf-8') as f:
    current_v2_stories = json.load(f)

current_v2_df = pd.DataFrame(current_v2_stories).copy()
current_v2_df['text'] = current_v2_df['text'].fillna('').astype(str)

def special_char_ratio(text):
    special = sum((not c.isalnum()) and (not c.isspace()) for c in text)
    return special / max(len(text), 1)

def normalize_special_text(text):
    text = str(text)
    # Remove HTML tags and URLs
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    # Remove repeated junk symbols but keep normal punctuation like ! ? . , ' " -
    text = re.sub(r'[@#$%^&*_+=|\/~`<>\[\]{}]{2,}', ' ', text)
    text = re.sub(r'[�☐☒□■◆]+', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'\s+([,.;:!?])', r'\1', text)
    return text.strip()

current_v2_df['special_ratio'] = current_v2_df['text'].apply(special_char_ratio)

# Corrupted records: obvious HTML / URL / mojibake / lots of junk symbols
corrupted_pattern = re.compile(r'<[^>]+>|https?://\S+|www\.\S+|[�☐☒□■◆]|[@#$%^&*_+=|\/~`<>\[\]{}]{3,}')
current_v2_df['is_corrupted_special'] = current_v2_df['text'].apply(lambda x: bool(corrupted_pattern.search(x)))

to_delete_df = current_v2_df[current_v2_df['is_corrupted_special']].copy()
to_normalize_df = current_v2_df[(current_v2_df['special_ratio'] >= 0.10) & (current_v2_df['special_ratio'] <= 0.20) & (~current_v2_df['is_corrupted_special'])].copy()
manual_check_df = current_v2_df[(current_v2_df['special_ratio'] > 0.20) & (~current_v2_df['is_corrupted_special'])].copy()

delete_ids = set(to_delete_df['unified_id'])
normalize_ids = set(to_normalize_df['unified_id'])
manual_check_ids = manual_check_df['unified_id'].tolist()

normalized_log = []
deleted_log = []
updated_records = []

for record in current_v2_stories:
    updated = dict(record)
    uid = updated['unified_id']

    if uid in delete_ids:
        deleted_log.append({
            'unified_id': uid,
            'title': updated.get('title', ''),
            'reason': 'corrupted_special',
        })
        continue

    if uid in normalize_ids:
        old_text = str(updated.get('text', ''))
        new_text = normalize_special_text(old_text)
        updated['text'] = new_text
        updated['word_count'] = len(new_text.split()) if new_text else 0
        normalized_log.append({
            'unified_id': uid,
            'title': updated.get('title', ''),
            'special_ratio': float(to_normalize_df.loc[to_normalize_df['unified_id'] == uid, 'special_ratio'].iloc[0]),
        })

    updated_records.append(updated)

with v2_path.open('w', encoding='utf-8') as f:
    json.dump(updated_records, f, ensure_ascii=False, indent=2)

with normalize_log_path.open('w', encoding='utf-8') as f:
    json.dump(normalized_log, f, ensure_ascii=False, indent=2)

with delete_log_path.open('w', encoding='utf-8') as f:
    json.dump(deleted_log, f, ensure_ascii=False, indent=2)

updated_v2_df = pd.DataFrame(updated_records)
updated_counts = (
    updated_v2_df
    .groupby(['category_v1', 'reading_level_key'])
    .size()
    .reset_index(name='book_count')
    .sort_values(['category_v1', 'reading_level_key'])
)

updated_pivot = (
    updated_counts
    .pivot(index='category_v1', columns='reading_level_key', values='book_count')
    .fillna(0)
    .astype(int)
)

print(f'Records normalized in V2 (special_ratio 0.10-0.20): {len(normalized_log)}')
print(f'Records deleted from V2 (corrupted special): {len(deleted_log)}')
print(f'Manual-check records (special_ratio > 0.20, not corrupted): {len(manual_check_ids)}')
print('Manual-check unified_ids:')
for uid in manual_check_ids:
    print(uid)

print(f'Overwrote V2 at {v2_path}')
print(f'Saved normalize log to {normalize_log_path}')
print(f'Saved delete log to {delete_log_path}')

display(pd.DataFrame(normalized_log).head(100))
display(pd.DataFrame(deleted_log).head(100))
display(manual_check_df[['unified_id', 'title', 'word_count', 'special_ratio']].sort_values('special_ratio', ascending=False))
display(updated_counts)
display(updated_pivot)


Records normalized in V2 (special_ratio 0.10-0.20): 71
Records deleted from V2 (corrupted special): 45
Manual-check records (special_ratio > 0.20, not corrupted): 10
Manual-check unified_ids:
storyweaver_16939
storyweaver_22557
storyweaver_52198
storyweaver_90549
storyweaver_136564
storyweaver_229847
storyweaver_266769
storyweaver_603270
storyweaver_605976
storyweaver_634986
Overwrote V2 at story_categories_v2.json
Saved normalize log to story_categories_v2_special_normalized.json
Saved delete log to story_categories_v2_deleted_corrupted_special.json


,unified_id,title,special_ratio
0,storyweaver_1658,Ammu's Puppy,0.107314
1,storyweaver_9758,Ice-cream delight,0.112903
2,storyweaver_12275,April fool.... have fun!,0.100075
3,storyweaver_12758,My Brother and Me,0.141949
4,storyweaver_14378,My Buddies....,0.158177
...,...,...,...
66,storyweaver_665072,Echoes of Childhood,0.121517
67,storyweaver_668752,The curious case of the missing socks,0.143396
68,storyweaver_670224,I like to play and learn,0.132308
69,storyweaver_673628,socialmedia,0.105991


,unified_id,title,reason
0,storyweaver_5554,The Theft,corrupted_special
1,storyweaver_6084,Friendly Neighbourhood Dog,corrupted_special
2,storyweaver_6932,Anita and Cindy,corrupted_special
3,storyweaver_8985,The Raja of Udaipur,corrupted_special
4,storyweaver_17952,Joke Corner 1,corrupted_special
5,storyweaver_18009,JOKE CORNER 4,corrupted_special
6,storyweaver_34426,How Heavy is Air?,corrupted_special
7,storyweaver_44402,Maths at the Mela,corrupted_special
8,storyweaver_53482,The Grand Super Hero Plan,corrupted_special
9,storyweaver_59369,Minni plays Bhatukali,corrupted_special


,unified_id,title,word_count,special_ratio
6032,storyweaver_603270,Deep Jungle,98,0.493802
6406,storyweaver_634986,the first olympiad exam,180,0.471500
6054,storyweaver_605976,One Wonderful Vacation,142,0.335345
3592,storyweaver_266769,Little Miss Ant,86,0.297659
3344,storyweaver_229847,Quiz - Count How Many?,192,0.279031
762,storyweaver_22557,Mitu's friends,112,0.275033
2400,storyweaver_136564,Say in a breath,89,0.274005
1365,storyweaver_52198,nina part 2,117,0.273932
1824,storyweaver_90549,JOKER,245,0.242700
611,storyweaver_16939,What is it?,104,0.206186


,category_v1,reading_level_key,book_count
0,Animals,level_1,977
1,Animals,level_2,1023
2,Animals,level_3,764
3,Daily Life,level_1,1117
4,Daily Life,level_2,1382
5,Daily Life,level_3,1342
6,Science & Knowledge,level_1,80
7,Science & Knowledge,level_2,112
8,Science & Knowledge,level_3,207


reading_level_key,level_1,level_2,level_3
category_v1,,,
Animals,977,1023,764
Daily Life,1117,1382,1342
Science & Knowledge,80,112,207


In [31]:
# Apply manual decisions for special-ratio review records in V2, then recalculate counts
v2_path = LIBRARY_DIR / 'story_categories_v2.json'
manual_log_path = LIBRARY_DIR / 'story_categories_v2_manual_special_actions.json'

delete_ids = {
    'storyweaver_52198',
    'storyweaver_90549',
    'storyweaver_605976',
    'storyweaver_634986',
}

review_ids = {
    'storyweaver_16939',
    'storyweaver_22557',
    'storyweaver_52198',
    'storyweaver_90549',
    'storyweaver_136564',
    'storyweaver_229847',
    'storyweaver_266769',
    'storyweaver_603270',
    'storyweaver_605976',
    'storyweaver_634986',
}

normalize_ids = review_ids - delete_ids

with v2_path.open(encoding='utf-8') as f:
    current_v2_stories = json.load(f)

def normalize_special_text(text):
    text = str(text)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    text = re.sub(r'[@#$%^&*_+=|\/~`<>\[\]{}]{2,}', ' ', text)
    text = re.sub(r'[�☐☒□■◆]+', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'\s+([,.;:!?])', r'\1', text)
    return text.strip()

updated_records = []
action_log = []
deleted_count = 0
normalized_count = 0

for record in current_v2_stories:
    uid = record.get('unified_id')

    if uid in delete_ids:
        deleted_count += 1
        action_log.append({
            'unified_id': uid,
            'title': record.get('title', ''),
            'action': 'delete',
        })
        continue

    updated = dict(record)
    if uid in normalize_ids:
        old_text = str(updated.get('text', ''))
        new_text = normalize_special_text(old_text)
        updated['text'] = new_text
        updated['word_count'] = len(new_text.split()) if new_text else 0
        normalized_count += 1
        action_log.append({
            'unified_id': uid,
            'title': updated.get('title', ''),
            'action': 'keep_normalize',
        })

    updated_records.append(updated)

with v2_path.open('w', encoding='utf-8') as f:
    json.dump(updated_records, f, ensure_ascii=False, indent=2)

with manual_log_path.open('w', encoding='utf-8') as f:
    json.dump(action_log, f, ensure_ascii=False, indent=2)

updated_v2_df = pd.DataFrame(updated_records)
updated_counts = (
    updated_v2_df
    .groupby(['category_v1', 'reading_level_key'])
    .size()
    .reset_index(name='book_count')
    .sort_values(['category_v1', 'reading_level_key'])
)

updated_pivot = (
    updated_counts
    .pivot(index='category_v1', columns='reading_level_key', values='book_count')
    .fillna(0)
    .astype(int)
)

print(f'Records normalized: {normalized_count}')
print(f'Records deleted: {deleted_count}')
print(f'Records remaining in V2: {len(updated_records)}')
print(f'Overwrote V2 at {v2_path}')
print(f'Saved manual action log to {manual_log_path}')
display(pd.DataFrame(action_log))
display(updated_counts)
display(updated_pivot)


Records normalized: 6
Records deleted: 4
Records remaining in V2: 7000
Overwrote V2 at story_categories_v2.json
Saved manual action log to story_categories_v2_manual_special_actions.json


,unified_id,title,action
0,storyweaver_16939,What is it?,keep_normalize
1,storyweaver_22557,Mitu's friends,keep_normalize
2,storyweaver_52198,nina part 2,delete
3,storyweaver_90549,JOKER,delete
4,storyweaver_136564,Say in a breath,keep_normalize
5,storyweaver_229847,Quiz - Count How Many?,keep_normalize
6,storyweaver_266769,Little Miss Ant,keep_normalize
7,storyweaver_603270,Deep Jungle,keep_normalize
8,storyweaver_605976,One Wonderful Vacation,delete
9,storyweaver_634986,the first olympiad exam,delete


,category_v1,reading_level_key,book_count
0,Animals,level_1,977
1,Animals,level_2,1023
2,Animals,level_3,764
3,Daily Life,level_1,1116
4,Daily Life,level_2,1381
5,Daily Life,level_3,1341
6,Science & Knowledge,level_1,80
7,Science & Knowledge,level_2,112
8,Science & Knowledge,level_3,206


reading_level_key,level_1,level_2,level_3
category_v1,,,
Animals,977,1023,764
Daily Life,1116,1381,1341
Science & Knowledge,80,112,206


In [32]:
# Check whether V2 contains books with unusually long sentences
v2_path = LIBRARY_DIR / 'story_categories_v2.json'

with v2_path.open(encoding='utf-8') as f:
    v2_stories = json.load(f)

sentence_df = pd.DataFrame(v2_stories).copy()
sentence_df['text'] = sentence_df['text'].fillna('').astype(str)

def split_sentences(text):
    parts = re.split(r'(?<=[.!?])\s+|\n+', text)
    return [p.strip() for p in parts if p.strip()]

def avg_sentence_length(text):
    sentences = split_sentences(text)
    if not sentences:
        return 0
    word_counts = [len(s.split()) for s in sentences]
    return sum(word_counts) / len(word_counts)

def max_sentence_length(text):
    sentences = split_sentences(text)
    if not sentences:
        return 0
    return max(len(s.split()) for s in sentences)

sentence_df['avg_sentence_len'] = sentence_df['text'].apply(avg_sentence_length)
sentence_df['max_sentence_len'] = sentence_df['text'].apply(max_sentence_length)

# Flag books that may be problematic for simpler reading use cases
suspect_sentence_df = sentence_df[(sentence_df['word_count'] >= 30) & ((sentence_df['avg_sentence_len'] > 20) | (sentence_df['max_sentence_len'] > 40))].copy()
suspect_sentence_df = suspect_sentence_df.sort_values(['avg_sentence_len', 'max_sentence_len'], ascending=[False, False])

print(f'Total V2 records checked: {len(sentence_df)}')
print(f'Suspected long-sentence records: {len(suspect_sentence_df)}')

display(suspect_sentence_df[['unified_id', 'title', 'category_v1', 'reading_level_key', 'word_count', 'avg_sentence_len', 'max_sentence_len']].head(100))

sentence_snippets = suspect_sentence_df[['unified_id', 'title', 'text']].copy()
sentence_snippets['text_snippet'] = sentence_snippets['text'].str.slice(0, 400)
display(sentence_snippets[['unified_id', 'title', 'text_snippet']].head(20))


Total V2 records checked: 7000
Suspected long-sentence records: 1837


,unified_id,title,category_v1,reading_level_key,word_count,avg_sentence_len,max_sentence_len
5112,storyweaver_480870,N P Singh,Daily Life,level_3,515,515.000000,515
4199,storyweaver_349998,DINO WORLD,Animals,level_3,298,298.000000,298
6292,storyweaver_627046,Mythical dragon,Animals,level_2,274,274.000000,274
6631,storyweaver_661627,Teke Teke -the horror legend of Japan,Daily Life,level_1,260,260.000000,260
722,storyweaver_21894,One Beautiful Tree,Science & Knowledge,level_2,249,249.000000,249
...,...,...,...,...,...,...,...
3683,storyweaver_280657,The boy how loves his grandma,Daily Life,level_2,70,70.000000,70
3096,storyweaver_200868,The magic carpet,Daily Life,level_3,552,69.000000,177
3695,storyweaver_281015,alien cat,Daily Life,level_1,138,69.000000,70
5733,storyweaver_561497,Introducing myself,Daily Life,level_3,69,69.000000,69


,unified_id,title,text_snippet
5112,storyweaver_480870,N P Singh,Once upon a time in a small village named Bons...
4199,storyweaver_349998,DINO WORLD,So this story is about me jesse a cute little ...
6292,storyweaver_627046,Mythical dragon,One day happy dragon was flying.he saw a gem 💎...
6631,storyweaver_661627,Teke Teke -the horror legend of Japan,maana jaata hai ki teke teke naam ki ek darawa...
722,storyweaver_21894,One Beautiful Tree,It is standing on the ground It is a green gre...
6734,storyweaver_673104,Donkey Learns a lesson,Long ago there lived a merchant Jim who dealt ...
3615,storyweaver_271702,never behave bad,One day two sister name Tina and small sister ...
6293,storyweaver_627057,Neenu the good girl,Neenu is naughty girl she doesn't share anythi...
6404,storyweaver_640108,friend ship of rishikrish and shrikanth,rishikrish and shrikanth born on 2016 and ther...
4637,storyweaver_419974,Stripes,once there lived a zebra who always used to as...


In [36]:
# Repair sentence splitting in V2 before checking long-sentence problems
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd

v2_path = LIBRARY_DIR / 'story_categories_v2.json'

with v2_path.open(encoding="utf-8") as f:
    v2_stories = json.load(f)

sentence_fix_df = pd.DataFrame(v2_stories).copy()
sentence_fix_df["text"] = sentence_fix_df["text"].fillna("").astype(str)

def split_sentences_fixed(text):
    text = str(text)
    text = re.sub(r"\s+", " ", text).strip()
    text = re.sub(r"\.{3,}", ".", text)
    text = re.sub(r"([.!?])\s+", r"\1|", text)
    text = re.sub(r"[\n\r]+", "|", text)
    text = re.sub(r"\|+", "|", text)
    sentences = [s.strip() for s in text.split("|") if s.strip()]
    return sentences

def sentence_stats(text):
    sentences = split_sentences_fixed(text)
    lens = []
    for s in sentences:
        words = re.findall(r"\b[a-zA-Z]+\b", s)
        if words:
            lens.append(len(words))
    if not lens:
        return pd.Series([0.0, 0, 0])
    return pd.Series([float(np.mean(lens)), int(max(lens)), int(len(lens))])

sentence_fix_df[["avg_sentence_len_fixed", "max_sentence_len_fixed", "sentence_count_fixed"]] = (
    sentence_fix_df["text"].apply(sentence_stats)
)

print(f"Total V2 records loaded: {len(sentence_fix_df)}")
display(
    sentence_fix_df[[
        "unified_id",
        "title",
        "reading_level_key",
        "word_count",
        "avg_sentence_len_fixed",
        "max_sentence_len_fixed",
        "sentence_count_fixed",
    ]].head(20)
)


Total V2 records loaded: 7000


,unified_id,title,reading_level_key,word_count,avg_sentence_len_fixed,max_sentence_len_fixed,sentence_count_fixed
0,storyweaver_21,Chuskit Goes to School!,level_3,1582,12.944000,35.0,125.0
1,storyweaver_33,"Not Now, Not Now!",level_1,177,17.700000,37.0,10.0
2,storyweaver_38,Counting on Moru,level_3,1960,12.135802,35.0,162.0
3,storyweaver_54,Susheela's Kolams,level_1,210,10.047619,21.0,21.0
4,storyweaver_67,"Sister, Sister, Where Does the Sun Go at Night?",level_3,1001,12.580247,31.0,81.0
5,storyweaver_80,Aunty Jui's Baby,level_2,623,9.521739,42.0,69.0
6,storyweaver_111,Little by Little,level_2,471,10.500000,30.0,46.0
7,storyweaver_127,The Missing Bat,level_3,555,12.400000,39.0,45.0
8,storyweaver_132,Daddy's Mo,level_2,416,11.729730,28.0,37.0
9,storyweaver_141,Going to a Wedding,level_2,285,5.836735,13.0,49.0


In [37]:
# Re-check long-sentence problems using repaired sentence splitting and level-specific thresholds
def long_sentence_flag(row):
    level = row["reading_level_key"]
    avg_len = row["avg_sentence_len_fixed"]
    max_len = row["max_sentence_len_fixed"]

    if level == "level_1":
        return (avg_len > 15) or (max_len > 35)
    elif level == "level_2":
        return (avg_len > 20) or (max_len > 50)
    elif level == "level_3":
        return (avg_len > 25) or (max_len > 70)
    return False

sentence_fix_df["long_sentence_flag"] = sentence_fix_df.apply(long_sentence_flag, axis=1)
sentence_fix_df["possible_missing_punctuation"] = (
    (sentence_fix_df["sentence_count_fixed"] <= 2)
    & (sentence_fix_df["word_count"] > 100)
)

suspect_long_sentence_df = sentence_fix_df[
    sentence_fix_df["long_sentence_flag"] | sentence_fix_df["possible_missing_punctuation"]
].copy()

suspect_long_sentence_df = suspect_long_sentence_df.sort_values(
    ["reading_level_key", "avg_sentence_len_fixed", "max_sentence_len_fixed"],
    ascending=[True, False, False],
)

print(f"Total V2 records checked: {len(sentence_fix_df)}")
print(f"Long-sentence flagged records: {int(sentence_fix_df['long_sentence_flag'].sum())}")
print(f"Possible missing-punctuation records: {int(sentence_fix_df['possible_missing_punctuation'].sum())}")
print(f"Total suspect records after repaired sentence splitting: {len(suspect_long_sentence_df)}")

display(
    suspect_long_sentence_df[[
        "unified_id",
        "title",
        "reading_level_key",
        "word_count",
        "avg_sentence_len_fixed",
        "max_sentence_len_fixed",
        "sentence_count_fixed",
        "long_sentence_flag",
        "possible_missing_punctuation",
    ]].head(100)
)

suspect_snippets = suspect_long_sentence_df[["unified_id", "title", "text"]].copy()
suspect_snippets["text_snippet"] = suspect_snippets["text"].str.slice(0, 400)
display(suspect_snippets[["unified_id", "title", "text_snippet"]].head(20))


Total V2 records checked: 7000
Long-sentence flagged records: 1295
Possible missing-punctuation records: 78
Total suspect records after repaired sentence splitting: 1301


,unified_id,title,reading_level_key,word_count,avg_sentence_len_fixed,max_sentence_len_fixed,sentence_count_fixed,long_sentence_flag,possible_missing_punctuation
6631,storyweaver_661627,Teke Teke -the horror legend of Japan,level_1,260,261.00,261.0,1.0,True,True
6734,storyweaver_673104,Donkey Learns a lesson,level_1,206,211.00,211.0,1.0,True,True
3615,storyweaver_271702,never behave bad,level_1,186,188.00,188.0,1.0,True,True
6293,storyweaver_627057,Neenu the good girl,level_1,184,186.00,186.0,1.0,True,True
2813,storyweaver_171674,holi is a holiday,level_1,139,140.00,140.0,1.0,True,True
...,...,...,...,...,...,...,...,...,...
4227,storyweaver_353331,Nali's great idea,level_1,186,47.75,153.0,4.0,True,False
5920,storyweaver_596944,A day in the sea,level_1,243,47.00,108.0,5.0,True,False
4349,storyweaver_369478,sad monkey,level_1,96,47.00,62.0,2.0,True,False
4433,storyweaver_381022,Tommy in dark nights,level_1,235,46.80,144.0,5.0,True,False


,unified_id,title,text_snippet
6631,storyweaver_661627,Teke Teke -the horror legend of Japan,maana jaata hai ki teke teke naam ki ek darawa...
6734,storyweaver_673104,Donkey Learns a lesson,Long ago there lived a merchant Jim who dealt ...
3615,storyweaver_271702,never behave bad,One day two sister name Tina and small sister ...
6293,storyweaver_627057,Neenu the good girl,Neenu is naughty girl she doesn't share anythi...
2813,storyweaver_171674,holi is a holiday,This is tinku and she always like to celebrate...
6291,storyweaver_626970,My House,This is my house ﻿I live here with my family W...
3682,storyweaver_280209,yellow ladybug,one day a yellow ladybug was going to school e...
4293,storyweaver_361704,SECRET 4,"A girl named tinni was with her friend peenie,..."
4609,storyweaver_416294,calm lady,there was once a girl that got lost on an aben...
6294,storyweaver_627063,Friends,Rinu rhino has no friends is doesn't talk to a...


In [38]:
# Further improve sentence splitting for records with possible_missing_punctuation == True
import re

repair_df = sentence_fix_df.copy()

def split_sentences_enhanced(text):
    text = str(text)
    text = text.replace(";", "; ")
    text = re.sub(r"\s+", " ", text).strip()
    text = re.sub(r"\.{3,}", ".", text)
    text = re.sub(r"([.!?;:])\s+", r"\1|", text)
    text = re.sub(r"\b(And|But|So|Then|Because|After|Before|When|While|If|Suddenly|Finally|Next|Later|Meanwhile|Once|Today|Tomorrow|First|Second|Third)\b", r"|\1", text)
    text = re.sub(r"\b(He|She|They|We|I|It|The|A|An|This|That|These|Those)\s+([A-Z][a-z]+)", r"|\1 \2", text)
    text = re.sub(r"\|+", "|", text)
    text = text.strip("| ")
    sentences = [s.strip() for s in text.split("|") if s.strip()]
    return sentences

def sentence_stats_enhanced(text):
    sentences = split_sentences_enhanced(text)
    lens = []
    for s in sentences:
        words = re.findall(r"\b[a-zA-Z]+\b", s)
        if words:
            lens.append(len(words))
    if not lens:
        return pd.Series([0.0, 0, 0])
    return pd.Series([float(np.mean(lens)), int(max(lens)), int(len(lens))])

needs_repair_mask = repair_df["possible_missing_punctuation"] == True
repair_df.loc[needs_repair_mask, ["avg_sentence_len_fixed", "max_sentence_len_fixed", "sentence_count_fixed"]] = (
    repair_df.loc[needs_repair_mask, "text"].apply(sentence_stats_enhanced).to_numpy()
)

repair_df["long_sentence_flag_after_repair"] = repair_df.apply(long_sentence_flag, axis=1)
repair_df["possible_missing_punctuation_after_repair"] = (
    (repair_df["sentence_count_fixed"] <= 2)
    & (repair_df["word_count"] > 100)
)

remaining_suspect_df = repair_df[
    repair_df["long_sentence_flag_after_repair"] | repair_df["possible_missing_punctuation_after_repair"]
].copy()

print(f"Records repaired in this pass: {int(needs_repair_mask.sum())}")
print(f"Possible missing punctuation before repair: {int(sentence_fix_df['possible_missing_punctuation'].sum())}")
print(f"Possible missing punctuation after repair: {int(repair_df['possible_missing_punctuation_after_repair'].sum())}")
print(f"Long-sentence flagged after repair: {int(repair_df['long_sentence_flag_after_repair'].sum())}")
print(f"Total suspect records after enhanced splitting: {len(remaining_suspect_df)}")

display(
    remaining_suspect_df[[
        "unified_id",
        "title",
        "reading_level_key",
        "word_count",
        "avg_sentence_len_fixed",
        "max_sentence_len_fixed",
        "sentence_count_fixed",
        "long_sentence_flag_after_repair",
        "possible_missing_punctuation_after_repair",
    ]].head(100)
)

remaining_snippets = remaining_suspect_df[["unified_id", "title", "text"]].copy()
remaining_snippets["text_snippet"] = remaining_snippets["text"].str.slice(0, 400)
display(remaining_snippets[["unified_id", "title", "text_snippet"]].head(20))


Records repaired in this pass: 78
Possible missing punctuation before repair: 78
Possible missing punctuation after repair: 56
Long-sentence flagged after repair: 1292
Total suspect records after enhanced splitting: 1298


,unified_id,title,reading_level_key,word_count,avg_sentence_len_fixed,max_sentence_len_fixed,sentence_count_fixed,long_sentence_flag_after_repair,possible_missing_punctuation_after_repair
1,storyweaver_33,"Not Now, Not Now!",level_1,177,17.700000,37.0,10.0,True,False
21,storyweaver_245,Rhino Charge,level_2,738,12.466667,52.0,60.0,True,False
36,storyweaver_638,Kallu's World 1 - In Big Trouble Again!,level_3,1983,13.925170,73.0,147.0,True,False
49,storyweaver_880,What If?,level_1,211,11.722222,38.0,18.0,True,False
87,storyweaver_2183,The Race!,level_2,135,17.375000,59.0,8.0,True,False
...,...,...,...,...,...,...,...,...,...
545,storyweaver_14405,Kato Clever and the Big Trouble,level_2,494,17.241379,54.0,29.0,True,False
553,storyweaver_14785,Ah! Football!,level_1,238,9.230769,41.0,26.0,True,False
559,storyweaver_14884,Elly and Ex,level_2,529,16.500000,67.0,32.0,True,False
561,storyweaver_14955,The race,level_1,271,12.681818,37.0,22.0,True,False


,unified_id,title,text_snippet
1,storyweaver_33,"Not Now, Not Now!","I asked Ajji, “May I please have some laddoos?..."
21,storyweaver_245,Rhino Charge,In the grasslands of a National Park called Ka...
36,storyweaver_638,Kallu's World 1 - In Big Trouble Again!,“Kallu get up! You’ll get late for school agai...
49,storyweaver_880,What If?,"My name is Shyam, I am ten years old. I am a l..."
87,storyweaver_2183,The Race!,"He was still full of energy, so he ran past th..."
88,storyweaver_2198,Richard's unlucky day,Once there was a boy named Richard and he alwa...
90,storyweaver_2245,The party,Toku was a jungle boy.He had animal friends. E...
91,storyweaver_2375,The train journey,"Latika, Vidvath and Shubhi were very excited t..."
93,storyweaver_2482,Chameleon Life,A Chameleon is color changing animal and it is...
99,storyweaver_2638,Amma's stories,Hooja Singh Hooja Singh was a intelligent man....


In [39]:
# Turn remaining long-sentence issues into a difficulty feature instead of deleting them
difficulty_df = repair_df.copy()

def sentence_difficulty(row):
    if row["possible_missing_punctuation_after_repair"]:
        return "punctuation_review"

    level = row["reading_level_key"]
    avg_len = row["avg_sentence_len_fixed"]
    max_len = row["max_sentence_len_fixed"]

    if level == "level_1":
        if avg_len > 25 or max_len > 70:
            return "too_long_for_level"
        elif avg_len > 15 or max_len > 35:
            return "long_but_acceptable"

    elif level == "level_2":
        if avg_len > 35 or max_len > 90:
            return "too_long_for_level"
        elif avg_len > 20 or max_len > 50:
            return "long_but_acceptable"

    elif level == "level_3":
        if avg_len > 45 or max_len > 130:
            return "too_long_for_level"
        elif avg_len > 25 or max_len > 70:
            return "long_but_acceptable"

    return "normal"

difficulty_df["sentence_difficulty_flag"] = difficulty_df.apply(sentence_difficulty, axis=1)

print("Sentence difficulty flag counts after enhanced splitting:")
display(difficulty_df["sentence_difficulty_flag"].value_counts(dropna=False).rename_axis("sentence_difficulty_flag").reset_index(name="count"))

manual_review_df = difficulty_df[difficulty_df["sentence_difficulty_flag"] == "too_long_for_level"].copy()
punctuation_review_df = difficulty_df[difficulty_df["sentence_difficulty_flag"] == "punctuation_review"].copy()

print(f"Needs manual review now (too_long_for_level): {len(manual_review_df)}")
print(f"Still likely sentence-splitting problems (punctuation_review): {len(punctuation_review_df)}")

display(
    manual_review_df[[
        "unified_id",
        "title",
        "reading_level_key",
        "word_count",
        "avg_sentence_len_fixed",
        "max_sentence_len_fixed",
        "sentence_count_fixed",
        "sentence_difficulty_flag",
    ]].head(100)
)

display(
    punctuation_review_df[[
        "unified_id",
        "title",
        "reading_level_key",
        "word_count",
        "avg_sentence_len_fixed",
        "max_sentence_len_fixed",
        "sentence_count_fixed",
        "sentence_difficulty_flag",
    ]].head(100)
)


Sentence difficulty flag counts after enhanced splitting:


,sentence_difficulty_flag,count
0,normal,5702
1,long_but_acceptable,904
2,too_long_for_level,338
3,punctuation_review,56


Needs manual review now (too_long_for_level): 338
Still likely sentence-splitting problems (punctuation_review): 56


,unified_id,title,reading_level_key,word_count,avg_sentence_len_fixed,max_sentence_len_fixed,sentence_count_fixed,sentence_difficulty_flag
88,storyweaver_2198,Richard's unlucky day,level_3,204,51.750000,79.0,4.0,too_long_for_level
99,storyweaver_2638,Amma's stories,level_1,535,40.933333,73.0,15.0,too_long_for_level
104,storyweaver_2747,When the Wind Blew Today...,level_1,68,35.500000,62.0,2.0,too_long_for_level
114,storyweaver_3408,the clever tortoise and the foolish fox,level_2,126,44.000000,71.0,3.0,too_long_for_level
173,storyweaver_5466,Naina ka pyaar Chanda Bhaiyaa,level_2,430,61.857143,225.0,7.0,too_long_for_level
...,...,...,...,...,...,...,...,...
2456,storyweaver_140477,THE TWO FUNNY CATS.,level_1,102,34.000000,74.0,3.0,too_long_for_level
2497,storyweaver_144607,THE EVIL CAKE,level_1,339,49.714286,94.0,7.0,too_long_for_level
2541,storyweaver_147382,The Dividing Lines,level_3,253,84.666667,158.0,3.0,too_long_for_level
2567,storyweaver_149541,Anny & Che,level_2,415,41.000000,79.0,11.0,too_long_for_level


,unified_id,title,reading_level_key,word_count,avg_sentence_len_fixed,max_sentence_len_fixed,sentence_count_fixed,sentence_difficulty_flag
350,storyweaver_9672,Beautiful silhouette,level_1,102,41.0,45.0,2.0,punctuation_review
607,storyweaver_16939,What is it?,level_1,104,108.0,108.0,1.0,punctuation_review
722,storyweaver_21894,One Beautiful Tree,level_2,249,250.0,250.0,1.0,punctuation_review
790,storyweaver_23471,Poor child-Rich child,level_2,121,63.0,119.0,2.0,punctuation_review
1290,storyweaver_48013,The Big Book of Boochandis,level_2,123,125.0,125.0,1.0,punctuation_review
1373,storyweaver_53451,The world has no rainbow,level_2,125,126.0,126.0,1.0,punctuation_review
1387,storyweaver_55651,An animal king contest,level_1,212,0.0,0.0,0.0,punctuation_review
2283,storyweaver_127961,One Day I Had a Good Dream,level_2,368,0.0,0.0,0.0,punctuation_review
2495,storyweaver_144373,My First Orchestra Concert,level_2,286,1.0,1.0,1.0,punctuation_review
2684,storyweaver_161522,healthy food vs junk food,level_2,105,52.5,59.0,2.0,punctuation_review


In [41]:
# Apply more aggressive sentence splitting only to punctuation_review records
punctuation_repair_df = difficulty_df.copy()
review_mask = punctuation_repair_df["sentence_difficulty_flag"] == "punctuation_review"

def split_sentences_aggressive(text):
    text = str(text)
    text = re.sub(r"\s+", " ", text).strip()
    text = re.sub(r"\.{3,}", ".", text)

    # Keep basic sentence boundaries first
    text = re.sub(r"([.!?;:])\s+", r"\1|", text)

    # Break before common sentence starters when punctuation is missing
    text = re.sub(r"\b(And|But|So|Then|Because|After|Before|When|While|If|Suddenly|Finally|Next|Later|Meanwhile|Once|Today|Tomorrow|First|Second|Third)\b", r"|\1", text)

    # Break at article/pronoun + capitalized noun patterns
    text = re.sub(r"\b(He|She|They|We|I|It|The|A|An|This|That|These|Those)\s+([A-Z][a-z]+)", r"|\1 \2", text)

    # Break after comma before likely new sentence starters
    text = re.sub(r",\s+(And|But|So|Then|He|She|They|We|I|It|The|A|An|This|That|These|Those)\b", r",|\1", text)

    # Break before dialogue-like capitalized continuations
    text = re.sub(r"\s+([A-Z][a-z]+\s+(said|asked|cried|shouted|replied))\b", r"|\1", text)

    text = re.sub(r"\|+", "|", text)
    text = text.strip("| ")
    sentences = [s.strip() for s in text.split("|") if s.strip()]
    return sentences

def sentence_stats_aggressive(text):
    sentences = split_sentences_aggressive(text)
    lens = []
    for s in sentences:
        words = re.findall(r"\b[a-zA-Z]+\b", s)
        if words:
            lens.append(len(words))
    if not lens:
        return pd.Series([0.0, 0, 0])
    return pd.Series([float(np.mean(lens)), int(max(lens)), int(len(lens))])

before_count = int(review_mask.sum())
punctuation_repair_df.loc[review_mask, ["avg_sentence_len_fixed", "max_sentence_len_fixed", "sentence_count_fixed"]] = (
    punctuation_repair_df.loc[review_mask, "text"].apply(sentence_stats_aggressive).to_numpy()
)

punctuation_repair_df["sentence_difficulty_flag_after_aggressive"] = punctuation_repair_df.apply(sentence_difficulty, axis=1)
after_review_mask = punctuation_repair_df["sentence_difficulty_flag_after_aggressive"] == "punctuation_review"
after_count = int(after_review_mask.sum())

print(f"punctuation_review before aggressive repair: {before_count}")
print(f"punctuation_review after aggressive repair: {after_count}")
print(f"Reduced by: {before_count - after_count}")

remaining_punctuation_review_df = punctuation_repair_df[after_review_mask].copy()
resolved_punctuation_review_df = punctuation_repair_df[review_mask & ~after_review_mask].copy()

print(f"Resolved punctuation_review records: {len(resolved_punctuation_review_df)}")
print(f"Still unresolved punctuation_review records: {len(remaining_punctuation_review_df)}")

display(
    remaining_punctuation_review_df[[
        "unified_id",
        "title",
        "reading_level_key",
        "word_count",
        "avg_sentence_len_fixed",
        "max_sentence_len_fixed",
        "sentence_count_fixed",
        "sentence_difficulty_flag_after_aggressive",
    ]].head(100)
)

remaining_punctuation_snippets = remaining_punctuation_review_df[["unified_id", "title", "text"]].copy()
remaining_punctuation_snippets["text_snippet"] = remaining_punctuation_snippets["text"].str.slice(0, 500)
display(remaining_punctuation_snippets[["unified_id", "title", "text_snippet"]].head(20))


punctuation_review before aggressive repair: 56
punctuation_review after aggressive repair: 56
Reduced by: 0
Resolved punctuation_review records: 0
Still unresolved punctuation_review records: 56


,unified_id,title,reading_level_key,word_count,avg_sentence_len_fixed,max_sentence_len_fixed,sentence_count_fixed,sentence_difficulty_flag_after_aggressive
350,storyweaver_9672,Beautiful silhouette,level_1,102,41.000000,45.0,2.0,punctuation_review
607,storyweaver_16939,What is it?,level_1,104,108.000000,108.0,1.0,punctuation_review
722,storyweaver_21894,One Beautiful Tree,level_2,249,250.000000,250.0,1.0,punctuation_review
790,storyweaver_23471,Poor child-Rich child,level_2,121,63.000000,119.0,2.0,punctuation_review
1290,storyweaver_48013,The Big Book of Boochandis,level_2,123,125.000000,125.0,1.0,punctuation_review
1373,storyweaver_53451,The world has no rainbow,level_2,125,126.000000,126.0,1.0,punctuation_review
1387,storyweaver_55651,An animal king contest,level_1,212,0.000000,0.0,0.0,punctuation_review
2283,storyweaver_127961,One Day I Had a Good Dream,level_2,368,0.000000,0.0,0.0,punctuation_review
2495,storyweaver_144373,My First Orchestra Concert,level_2,286,1.000000,1.0,1.0,punctuation_review
2684,storyweaver_161522,healthy food vs junk food,level_2,105,52.500000,59.0,2.0,punctuation_review


,unified_id,title,text_snippet
350,storyweaver_9672,Beautiful silhouette,"can you spot the silhouette of, 1) A lonely an..."
607,storyweaver_16939,What is it?,"“La-la, la, la, la, la, la…” “Is it from somew..."
722,storyweaver_21894,One Beautiful Tree,It is standing on the ground It is a green gre...
790,storyweaver_23471,Poor child-Rich child,"They are all children,although different, all ..."
1290,storyweaver_48013,The Big Book of Boochandis,"Boochandis are everywhere Boochandis are here,..."
1373,storyweaver_53451,The world has no rainbow,"The world is a duck bro, we have a Trump bro, ..."
1387,storyweaver_55651,An animal king contest,"어느날, 동물의 왕 치타가 있었어요. 하지만 동물의 왕 치타는 나이를 너무 많이 먹..."
2283,storyweaver_127961,One Day I Had a Good Dream,자전거를 좋아하는 모든 사람들에게 이 책을 바칩니다. 모두들 재미있게 읽어주시길 바...
2495,storyweaver_144373,My First Orchestra Concert,이사벨은 여덟 살이에요. 이사벨은 바이올린 켜는 것을 많이 좋아해요. 그래서 바이올...
2684,storyweaver_161522,healthy food vs junk food,Once in a town lived two friends named sam and...


In [42]:
# Delete the 56 unresolved punctuation_review records directly from V2
import json
from pathlib import Path

v2_path = LIBRARY_DIR / 'story_categories_v2.json'
deleted_log_path = LIBRARY_DIR / 'story_categories_v2_deleted_punctuation_review.json'

with v2_path.open(encoding="utf-8") as f:
    current_v2_records = json.load(f)

delete_ids = set(remaining_punctuation_review_df["unified_id"].tolist())

deleted_records = [record for record in current_v2_records if record.get("unified_id") in delete_ids]
kept_records = [record for record in current_v2_records if record.get("unified_id") not in delete_ids]

with deleted_log_path.open("w", encoding="utf-8") as f:
    json.dump(deleted_records, f, ensure_ascii=False, indent=2)

with v2_path.open("w", encoding="utf-8") as f:
    json.dump(kept_records, f, ensure_ascii=False, indent=2)

updated_v2_df = pd.DataFrame(kept_records).copy()
updated_v2_df["word_count"] = pd.to_numeric(updated_v2_df["word_count"], errors="coerce")

updated_counts = (
    updated_v2_df.groupby(["category_v1", "reading_level_key"])
    .size()
    .reset_index(name="book_count")
    .sort_values(["category_v1", "reading_level_key"])
)

updated_pivot = updated_counts.pivot(index="category_v1", columns="reading_level_key", values="book_count").fillna(0).astype(int)

print(f"Deleted punctuation_review records from V2: {len(deleted_records)}")
print(f"Remaining V2 records: {len(kept_records)}")
print(f"Saved deletion log to: {deleted_log_path}")

display(updated_counts)
display(updated_pivot)


Deleted punctuation_review records from V2: 56
Remaining V2 records: 6944
Saved deletion log to: story_categories_v2_deleted_punctuation_review.json


,category_v1,reading_level_key,book_count
0,Animals,level_1,965
1,Animals,level_2,1017
2,Animals,level_3,758
3,Daily Life,level_1,1098
4,Daily Life,level_2,1371
5,Daily Life,level_3,1339
6,Science & Knowledge,level_1,80
7,Science & Knowledge,level_2,110
8,Science & Knowledge,level_3,206


reading_level_key,level_1,level_2,level_3
category_v1,,,
Animals,965,1017,758
Daily Life,1098,1371,1339
Science & Knowledge,80,110,206


In [43]:
# Final full scan for any remaining non-English content in current V2
import json
import re
from pathlib import Path

v2_path = LIBRARY_DIR / 'story_categories_v2.json'

with v2_path.open(encoding="utf-8") as f:
    final_v2_records = json.load(f)

final_v2_df = pd.DataFrame(final_v2_records).copy()
final_v2_df["title"] = final_v2_df["title"].fillna("").astype(str)
final_v2_df["text"] = final_v2_df["text"].fillna("").astype(str)

non_english_patterns = {
    "hangul": re.compile(r"[\u1100-\u11FF\u3130-\u318F\uAC00-\uD7AF]"),
    "hiragana_katakana": re.compile(r"[\u3040-\u30FF]"),
    "cjk": re.compile(r"[\u3400-\u4DBF\u4E00-\u9FFF]"),
    "devanagari": re.compile(r"[\u0900-\u097F]"),
    "bengali": re.compile(r"[\u0980-\u09FF]"),
    "gurmukhi": re.compile(r"[\u0A00-\u0A7F]"),
    "gujarati": re.compile(r"[\u0A80-\u0AFF]"),
    "oriya": re.compile(r"[\u0B00-\u0B7F]"),
    "tamil": re.compile(r"[\u0B80-\u0BFF]"),
    "telugu": re.compile(r"[\u0C00-\u0C7F]"),
    "kannada": re.compile(r"[\u0C80-\u0CFF]"),
    "malayalam": re.compile(r"[\u0D00-\u0D7F]"),
    "thai": re.compile(r"[\u0E00-\u0E7F]"),
    "arabic": re.compile(r"[\u0600-\u06FF]"),
    "hebrew": re.compile(r"[\u0590-\u05FF]"),
    "cyrillic": re.compile(r"[\u0400-\u04FF]"),
    "greek": re.compile(r"[\u0370-\u03FF]"),
}

def detect_non_english_scripts(text):
    text = str(text)
    hits = []
    for script_name, pattern in non_english_patterns.items():
        if pattern.search(text):
            hits.append(script_name)
    return hits

final_v2_df["title_non_english_scripts"] = final_v2_df["title"].apply(detect_non_english_scripts)
final_v2_df["text_non_english_scripts"] = final_v2_df["text"].apply(detect_non_english_scripts)
final_v2_df["all_non_english_scripts"] = final_v2_df.apply(
    lambda row: sorted(set(row["title_non_english_scripts"] + row["text_non_english_scripts"])),
    axis=1,
)
final_v2_df["has_non_english_content"] = final_v2_df["all_non_english_scripts"].apply(bool)

remaining_non_english_df = final_v2_df[final_v2_df["has_non_english_content"]].copy()
remaining_non_english_df["text_snippet"] = remaining_non_english_df["text"].str.slice(0, 400)

script_counter = {}
for scripts in remaining_non_english_df["all_non_english_scripts"]:
    for script_name in scripts:
        script_counter[script_name] = script_counter.get(script_name, 0) + 1

script_counts_df = pd.DataFrame(
    sorted(script_counter.items(), key=lambda x: (-x[1], x[0])),
    columns=["script_name", "record_count"],
)

print(f"Total current V2 records checked: {len(final_v2_df)}")
print(f"Records with remaining non-English content: {len(remaining_non_english_df)}")

if len(script_counts_df) > 0:
    display(script_counts_df)
else:
    print("No remaining non-English scripts detected in current V2.")

display(
    remaining_non_english_df[[
        "unified_id",
        "title",
        "reading_level_key",
        "word_count",
        "title_non_english_scripts",
        "text_non_english_scripts",
        "all_non_english_scripts",
    ]].head(100)
)

display(remaining_non_english_df[["unified_id", "title", "text_snippet"]].head(20))


Total current V2 records checked: 6944
Records with remaining non-English content: 2


,script_name,record_count
0,hangul,2


,unified_id,title,reading_level_key,word_count,title_non_english_scripts,text_non_english_scripts,all_non_english_scripts
1036,storyweaver_36395,Moonriver,level_2,179,[],[hangul],[hangul]
1380,storyweaver_55634,Dog,level_1,75,[],[hangul],[hangul]


,unified_id,title,text_snippet
1036,storyweaver_36395,Moonriver,"""Here for you."" He gave me a Clematis with dir..."
1380,storyweaver_55634,Dog,멍청한 강아지가 있습니다. 그 강아지는 너무 바보 같아. 1 + 1도 몰랐어. 그 ...


In [44]:
# Delete all remaining non-English records from current V2
import json
from pathlib import Path

v2_path = LIBRARY_DIR / 'story_categories_v2.json'
deleted_log_path = LIBRARY_DIR / 'story_categories_v2_deleted_remaining_non_english.json'

with v2_path.open(encoding="utf-8") as f:
    current_v2_records = json.load(f)

delete_ids = set(remaining_non_english_df["unified_id"].tolist())

deleted_records = [record for record in current_v2_records if record.get("unified_id") in delete_ids]
kept_records = [record for record in current_v2_records if record.get("unified_id") not in delete_ids]

with deleted_log_path.open("w", encoding="utf-8") as f:
    json.dump(deleted_records, f, ensure_ascii=False, indent=2)

with v2_path.open("w", encoding="utf-8") as f:
    json.dump(kept_records, f, ensure_ascii=False, indent=2)

print(f"Deleted remaining non-English records from V2: {len(deleted_records)}")
print(f"Remaining V2 records after non-English cleanup: {len(kept_records)}")
print(f"Saved deletion log to: {deleted_log_path}")


Deleted remaining non-English records from V2: 2
Remaining V2 records after non-English cleanup: 6942
Saved deletion log to: story_categories_v2_deleted_remaining_non_english.json


In [45]:
# Final stats for the current cleaned V2
import json
from pathlib import Path

final_v2_path = LIBRARY_DIR / 'story_categories_v2.json'

with final_v2_path.open(encoding="utf-8") as f:
    final_v2_records = json.load(f)

final_stats_df = pd.DataFrame(final_v2_records).copy()
final_stats_df["word_count"] = pd.to_numeric(final_stats_df["word_count"], errors="coerce")

final_counts = (
    final_stats_df.groupby(["category_v1", "reading_level_key"])
    .size()
    .reset_index(name="book_count")
    .sort_values(["category_v1", "reading_level_key"])
)

final_pivot = final_counts.pivot(index="category_v1", columns="reading_level_key", values="book_count").fillna(0).astype(int)

print(f"Final current V2 record count: {len(final_stats_df)}")
display(final_counts)
display(final_pivot)


Final current V2 record count: 6942


,category_v1,reading_level_key,book_count
0,Animals,level_1,964
1,Animals,level_2,1017
2,Animals,level_3,758
3,Daily Life,level_1,1098
4,Daily Life,level_2,1370
5,Daily Life,level_3,1339
6,Science & Knowledge,level_1,80
7,Science & Knowledge,level_2,110
8,Science & Knowledge,level_3,206


reading_level_key,level_1,level_2,level_3
category_v1,,,
Animals,964,1017,758
Daily Life,1098,1370,1339
Science & Knowledge,80,110,206
